[Markdown Cell]

# Rebuild Phase 2 — B / lora

This notebook starts a **new NER fine-tuning experiment from your verified own Phase 1 checkpoint**. It does not need any old Phase 2 task output. The default RUN_MODE='develop' trains on the original 60 public training documents, selects by strict entity Macro-F1 on the ten validation documents, and exports learned weights plus predictions for all 35 test IDs.

**Add Input:** original prepared Phase 2 data, complete own Phase 1 B output, and matching tokenizer.json/metadata.json. A/B use the 200M preprocessing pair, C uses 100M, D uses 25M. These are token-budget names. The selected current foundations are A=4, B=6, C=2 and D=1 completed LM epochs. Keep ner_run_dir=None. A downloaded notebook or tokenized corpus alone cannot replace model checkpoint tensors.

For B distillation, run notebook 01 first, save its complete output, and attach its development/ NER task as the larger own A teacher. Alternatively attach complete own Phase 1 A as well so the runtime can train a teacher. This is actual teacher/student learning, with no external pretrained model or test gold.

Select **T4 x2**, then Run All. Development has a 90-minute training safety limit; a paused run is recoverable with the same whole output and settings. Save Version -> Save & Run All creates the persistent output needed by the later final/experiment notebooks.

This rebuild uses the repaired, locally verified runtime and the same adaptation/head presets as the corresponding repaired baseline notebook. It is a fresh execution, not recovery of a missing historical tensor file; exact old F1 is not promised. Keep the new best learned output for subsequent controlled experiments.


[Markdown Cell]

## 1. Environment

In [1]:
# [Code Cell]
import importlib.metadata as package_metadata
import os
from pathlib import Path
import subprocess
import sys
import json
import torch

needed=[]
for package,requirement in [('tokenizers','tokenizers>=0.20,<1'),('tqdm','tqdm>=4.66,<5')]:
    try:
        package_metadata.version(package)
    except package_metadata.PackageNotFoundError:
        needed.append(requirement)
if needed:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*needed])
os.environ['TOKENIZERS_PARALLELISM']='false'
print('Python:',sys.version.split()[0],'PyTorch:',torch.__version__)
print('Visible GPUs:',torch.cuda.device_count())


Python: 3.13.15 PyTorch: 2.11.0+cu128
Visible GPUs: 2


[Markdown Cell]

## 2. Modes, continuation and exact input selectors

In [2]:
# [Code Cell]
FAMILY='b'
METHOD='lora'
RUN_MODE='develop' # auto, develop, inference, refit70, develop_then_refit.
EXPERIMENT_ID='Rebuild_'+FAMILY.upper()+'_'+METHOD
OUTPUT_ROOT='/kaggle/working/cs781_phase2_rebuild/'+EXPERIMENT_ID
DEVELOPMENT_EPOCHS=15
ADDITIONAL_EPOCHS=0 # Completed repaired DEVELOPMENT output only.
TARGET_TOTAL_EPOCHS=None # Alternative exact repaired-development extension.
CHARACTER_CONTEXT=False # Studies test this; keep false for measured baseline.
INPUTS=dict(prepared_dir=None,parent_run_dir=None,ner_run_dir=None,
    tokenizer_path=None,metadata_path=None,teacher_run_dir=None,
    resume_development_dir=None,teacher_parent_run_dir=None)
KNOWN_PARENT_SHA256={'a': '0746fa1981f804a246b65d82f1d13e35c3a6bdf554bfff69df26a35dd1d5d131', 'b': 'e69a9d37dc90878244c6a0deb4a433caab0d03798d6a877e48873ca391cb6875', 'c': '448551eabcb94b369fc445c6a2da0386580dd6f6756c7706a1e560c0283b3fe1', 'd': '8330dc8645c5d833858128d397de9692996bcb278b2c1113e1378a57b4e84091'}
EXPECTED_PARENT_SHA256=KNOWN_PARENT_SHA256[FAMILY]
ALLOW_TEACHER_BOOTSTRAP=True
STUDY_BUDGET_HOURS=2.5
REFIT_STUDY_WINNER=False # Refit once recipe selection is finished.
Path(OUTPUT_ROOT).mkdir(parents=True,exist_ok=True)


[Markdown Cell]

### Inspect inputs before training

A Phase 1 path contains train_config.json. A NER/teacher task path contains job.json. Set an explicit selector to its printed /kaggle/input path if multiple versions are attached. The absence of a NER task is supported by auto/develop training.

In [3]:
# [Code Cell]
for pattern in ('train_config.json','job.json','prepared_manifest.json','phase2_prepared_*.zip','tokenizer.json'):
    for path in sorted(Path('/kaggle/input').rglob(pattern)):
        print(pattern,':',path)


train_config.json : /kaggle/input/notebooks/shubhamjoshi1806/modelb-decoder/model_b_extended_run/train_config.json
prepared_manifest.json : /kaggle/input/notebooks/shubhamjoshi1806/p2-data-audit-and-split/cs781_phase2_data/prepared_7ded046ba3e8/prepared_manifest.json
phase2_prepared_*.zip : /kaggle/input/notebooks/shubhamjoshi1806/p2-data-audit-and-split/cs781_phase2_data/phase2_prepared_7ded046ba3e8.zip
tokenizer.json : /kaggle/input/datasets/shubhamjoshi1806/dataset-tokenized-capstone/data-and-tokenizer-output (1)/kaggle/working/downloaded_output/tokenizer.json


[Markdown Cell]

## 3. Preserve saved small dependencies

In [4]:
# [Code Cell]
pins=[name+'=='+version for name,version in [('tokenizers','0.23.1'),('numpy','2.1.3')]
      if package_metadata.version(name)!=version]
if pins:subprocess.check_call([sys.executable,'-m','pip','install','-q',*pins])
print('Keep the recorded Kaggle PyTorch build for actual task/teacher replay and recovery.')


Keep the recorded Kaggle PyTorch build for actual task/teacher replay and recovery.


[Markdown Cell]

## 4. What changes, and how we measure it

Auto uses an existing completed held-out task when supplied. Otherwise it trains supervised development from own Phase 1, selects the best real checkpoint by strict entity Macro-F1, and refits a fresh task model on 70 public Train documents. Develop mode exports the held-out task without final refit and supports exact continuation of repaired development outputs. Inference requires a task-trained state; language-model weights alone cannot predict learned NER labels.

The optional new **character BiGRU** learns both directions across original characters within a window. Its zero residual projection starts at the common control function; packed lengths prevent padding from contaminating valid characters. It retains the required backbone/adaptation method. It adds learned context beyond the existing nine-character convolution head, rather than an entity dictionary or answer rules.

\[
h_{char}=h_{local}+P\,\operatorname{BiGRU}(h_{local}),\quad P_{init}=0,
\qquad L=L_{BIO}+
rac{\lambda}{2}(L_{start}+L_{end}).
\]

Study candidates retain the same public split/method and each cover all assigned characters. They test a context/boundary bundle and a stronger capacity/optimization bundle; they are not isolated single-factor ablations or proven improvements. Exact teacher/student KL transfers BIO probabilities on training characters only. [NER context background](https://aclanthology.org/N16-1030/), [distillation](https://arxiv.org/abs/1503.02531).

Calibration uses released validation only. Its cross-fitting is conditional on prior checkpoint selection, not an unbiased model test. After a 70-document refit there is no held-out validation metric. Source/parent/data/optimizer identities are preserved; existing older checkpoints are used as verified learned candidates, never silently migrated optimizer states.

In [5]:
# [Code Cell]
module_parts={}
def append_module(name: str,source: str) -> None:
    module_parts.setdefault(name,[]).append(source)


[Markdown Cell]

### __init__.py

Explicit module for bounded, recoverable factor comparisons.

[Markdown Cell]

Module imports and definitions — source part 1

In [6]:
# [Code Cell]
append_module('__init__.py', r'''"""Controlled legal NER studies on four own foundations and six adaptation methods."""
''')


[Markdown Cell]

### adaptation.py

These modular wrappers implement genuinely distinct trainable methods. Frozen weights remain FP32; zero-start adapter updates preserve the base function initially.

[Markdown Cell]

Explicit additive and multiplicative low-rank layers — source part 1

In [7]:
# [Code Cell]
append_module('adaptation.py', r'''# CELL: Explicit additive and multiplicative low-rank layers
import math
import torch
from torch import nn, Tensor
from torch.nn import functional as F

class LowRankLinear(nn.Module):
    """x[...,in] -> y[...,out], retaining frozen FP32 pretrained base weights.

    LoRA: W0 x + alpha/r B A x.
    LoRMA+: (I + alpha/r B A) W0 x; A consumes output features.
    Bias remains outside the multiplicative weight update.
    """
    def __init__(self, base: nn.Linear, rank: int, alpha: float, multiplicative: bool):
        super().__init__(); self.base=base; self.scale=alpha/rank
        self.multiplicative=multiplicative
        width=base.out_features if multiplicative else base.in_features
        self.low_rank_a=nn.Parameter(torch.empty(rank,width))
        self.low_rank_b=nn.Parameter(torch.zeros(base.out_features,rank))
        nn.init.kaiming_uniform_(self.low_rank_a,a=math.sqrt(5))
        self.base.requires_grad_(False)

    def forward(self, x: Tensor) -> Tensor:
        y=F.linear(x,self.base.weight,None)
        inputs=y if self.multiplicative else x
        y=y+self.scale*F.linear(F.linear(inputs,self.low_rank_a),self.low_rank_b)
        return y if self.base.bias is None else y+self.base.bias

class ResidualAdapter(nn.Module):
    """Frozen Transformer block + trainable normalized bottleneck residual.

    hidden[B,T,D], original mask -> hidden[B,T,D]. Zero up-projection starts
    at the exact original block function; the surrounding GRUs remain intact.
    """
    def __init__(self, base: nn.Module, width: int, bottleneck: int):
        super().__init__(); self.base=base; base.requires_grad_(False)
        self.adapter_norm=nn.LayerNorm(width)
        self.adapter_down=nn.Linear(width,bottleneck)
        self.adapter_up=nn.Linear(bottleneck,width)
        nn.init.zeros_(self.adapter_up.weight); nn.init.zeros_(self.adapter_up.bias)

    def forward(self, hidden: Tensor, allowed: Tensor | None) -> Tensor:
        hidden=self.base(hidden,allowed)
        return hidden+self.adapter_up(F.gelu(self.adapter_down(self.adapter_norm(hidden))))

''')


[Markdown Cell]

Apply one declared method, with an auditable list of adapted modules — source part 1

In [8]:
# [Code Cell]
append_module('adaptation.py', r'''# CELL: Apply one declared method, with an auditable list of adapted modules
def apply_adaptation(backbone: nn.Module, cfg, width: int) -> list[str]:
    changed=[]
    if cfg.method in ('full','distillation'):
        backbone.requires_grad_(True)
        if cfg.family=='b':
            for module in (backbone.decoder,backbone.target_positions,backbone.decoder_norm):
                module.requires_grad_(False)  # NER uses encode(), not generation decode().
        return ['all active backbone weights']
    backbone.requires_grad_(False)
    if cfg.method in ('lora','lorma'):
        roots=[('encoder',backbone.encoder)] if cfg.family=='b' else [('',backbone)]
        for prefix,root in roots:
            targets=[(name,module) for name,module in root.named_modules() if isinstance(module,nn.Linear)]
            for name,module in targets:
                parent_name,_,child=name.rpartition('.')
                parent=root.get_submodule(parent_name) if parent_name else root
                setattr(parent,child,LowRankLinear(module,cfg.adapter_rank,cfg.adapter_alpha,cfg.method=='lorma'))
                changed.append((prefix+'.'+name).strip('.'))
    elif cfg.method=='adapters':
        for s,stage in enumerate(backbone.stages):
            for b,block in enumerate(stage.blocks):
                stage.blocks[b]=ResidualAdapter(block,width,cfg.bottleneck_width)
                changed.append(f'stages.{s}.blocks.{b}')
    elif cfg.method=='partial':
        if cfg.unfreeze_top_layers>len(backbone.layers):
            raise ValueError('unfreeze_top_layers exceeds this parent architecture')
        for index in range(len(backbone.layers)-cfg.unfreeze_top_layers,len(backbone.layers)):
            backbone.layers[index].requires_grad_(True); changed.append(f'layers.{index}')
        backbone.final_norm.requires_grad_(True); changed.append('final_norm')
    if not changed:
        raise ValueError('Adaptation did not select any original backbone modules')
    return changed
''')


[Markdown Cell]

### artifacts.py

Find the successful prepared output by its exact split/artifact hashes, including nested Kaggle exports or the small prepared ZIP.

[Markdown Cell]

Discover the exact successful Phase 2 data bundle — source part 1

In [9]:
# [Code Cell]
append_module('artifacts.py', r'''# CELL: Discover the exact successful Phase 2 data bundle
import importlib
import json
from pathlib import Path
import shutil
import sys
import zipfile
import torch
from tokenizers import Tokenizer
from tqdm.auto import tqdm
from .config import NERConfig
from .io_utils import atomic_json, ensure_space, identity, read_json, records, sha256_file

def prepared_input(cfg: NERConfig) -> Path:
    roots = [Path(cfg.prepared_dir)] if cfg.prepared_dir else list(Path(cfg.input_root).rglob('prepared_manifest.json'))
    candidates = []
    for value in roots:
        path = value / 'prepared_manifest.json' if value.is_dir() else value
        if path.is_file() and path.suffix == '.json':
            manifest = read_json(path)
            if manifest.get('identity') == cfg.expected_prepared_identity:
                candidates.append(path.parent)
    if not candidates:
        archives = [Path(cfg.prepared_dir)] if cfg.prepared_dir else list(Path(cfg.input_root).rglob('phase2_prepared_*.zip'))
        for path in archives:
            if not path.is_file() or path.suffix != '.zip':
                continue
            with zipfile.ZipFile(path) as archive:
                if 'prepared_manifest.json' not in archive.namelist():
                    continue
                manifest = json.loads(archive.read('prepared_manifest.json'))
                if manifest.get('identity') != cfg.expected_prepared_identity:
                    continue
                names = list(manifest['artifacts']) + ['prepared_manifest.json']
                if len(set(archive.namelist())) != len(archive.namelist()):
                    raise ValueError('Duplicate members in prepared ZIP.')
                if any('/' in name or '\\' in name or name in ('.', '..') for name in names):
                    raise ValueError('Unsafe prepared ZIP member.')
                destination = Path(cfg.run_dir) / 'prepared_input'; destination.mkdir(parents=True, exist_ok=True)
                required = sum(archive.getinfo(name).file_size for name in names)
                ensure_space(destination, required, cfg.min_free_bytes)
                for name in names:
                    with archive.open(name) as source, (destination / name).open('wb') as target:
                        shutil.copyfileobj(source, target, 8 << 20)
                candidates.append(destination)
                break
    if not candidates:
        raise FileNotFoundError('Attach the saved successful Phase 2 preparation output or phase2_prepared_7ded046ba3e8.zip. The .ipynb alone contains no prepared data.')
''')


[Markdown Cell]

Discover the exact successful Phase 2 data bundle — source part 2

In [10]:
# [Code Cell]
append_module('artifacts.py', r'''    root = candidates[0]
    manifest = read_json(root / 'prepared_manifest.json')
    if manifest.get('status') != 'complete' or manifest.get('split_sha256') != cfg.expected_split_sha256:
        raise ValueError('Prepared output is incomplete or uses a different split.')
    for name, receipt in tqdm(manifest['artifacts'].items(), desc='Verify prepared artifacts'):
        path = root / name
        if path.parent.resolve() != root.resolve() or not path.is_file():
            raise ValueError(f'Missing/unsafe prepared file: {name}')
        if path.stat().st_size != receipt['bytes'] or sha256_file(path) != receipt['sha256']:
            raise ValueError(f'Prepared artifact checksum mismatch: {name}')
    split = read_json(root / 'split_manifest.json')
    if identity(split['ids']) != cfg.expected_split_sha256:
        raise ValueError('Prepared split IDs changed.')
    labels = manifest['labels']
    if len(labels) != 12 or len(set(labels)) != 12:
        raise ValueError('Expected the twelve official labels.')
    for partition in ('train', 'validation', 'test'):
        rows = list(records(root / (partition + '.jsonl')))
        if [row['id'] for row in rows] != split['ids'][partition]:
            raise ValueError(f'Document IDs differ from the split: {partition}')
        if partition == 'test' and any('entities' in row for row in rows):
            raise ValueError('Test annotations must not be present.')
    return root

''')


[Markdown Cell]

### calibration.py

Public validation learns confidence filtering; conditional leave-one-document-out evidence gates per-label thresholds.

[Markdown Cell]

Cross-fitted confidence calibration from released validation labels only — source part 1

In [11]:
# [Code Cell]
append_module('calibration.py', r'''# CELL: Cross-fitted confidence calibration from released validation labels only
"""Learn filtering thresholds; never alter entity text, offsets or predicted types."""
import math
from .metrics import metrics,filter_spans

GRID=(0.0,.25,.4,.5,.65,.8,.9,.95)


def apply_policy(scored: dict,policy: dict) -> dict:
    """Original model spans/confidences -> retained triples under frozen policy."""
    global_threshold=policy['global_threshold'];thresholds=policy.get('per_label_thresholds',{})
    if not 0<=global_threshold<=1 or any(not 0<=t<=1 for t in thresholds.values()):
        raise ValueError('Invalid learned confidence thresholds')
    result={}
    for key,rows in scored.items():
        result[key]=[]
        for row in rows:
            confidence=float(row['confidence'])
            if not math.isfinite(confidence) or not 0<=confidence<=1:raise ValueError('Invalid model confidence')
            if confidence>=thresholds.get(row['span'][2],global_threshold):result[key].append(row['span'])
    return result


def fit_policies(scored: dict,gold: dict,labels: list[str],fallback: float) -> tuple[dict,dict]:
    """Fit global and label thresholds. No document identifiers become features."""
    if set(scored)!=set(gold) or not gold:raise ValueError('Calibration needs the original held-out IDs')
    grid=tuple(sorted(set((*GRID,float(fallback)))))
    reports={t:metrics(gold,filter_spans(scored,t),labels) for t in grid}
    global_t=max(grid,key=lambda t:(reports[t]['strict_macro_f1'],-abs(t-fallback),t))
    per_label={label:max(grid,key=lambda t:(reports[t]['per_label'][label]['f1'],-abs(t-global_t),t)) for label in labels}
    return {'kind':'global','global_threshold':global_t}, {'kind':'per_label','global_threshold':global_t,'per_label_thresholds':per_label}


def calibrate(scored: dict,gold: dict,labels: list[str],fallback: float,minimum_gain: float=.01) -> dict:
    """LOO threshold fits protect each held-out document's calibration labels.

    Checkpoints were already selected on validation, so this is conditional
    calibration evidence, not an unbiased estimate of model generalization.
    """
    if len(gold)<2:raise ValueError('At least two validation documents are required')
    global_oof,label_oof={},{}
    for heldout in gold:
        training_gold={key:value for key,value in gold.items() if key!=heldout}
        training_scores={key:value for key,value in scored.items() if key!=heldout}
        global_policy,label_policy=fit_policies(training_scores,training_gold,labels,fallback)
        global_oof[heldout]=apply_policy({heldout:scored[heldout]},global_policy)[heldout]
        label_oof[heldout]=apply_policy({heldout:scored[heldout]},label_policy)[heldout]
    global_report=metrics(gold,global_oof,labels);label_report=metrics(gold,label_oof,labels)
''')


[Markdown Cell]

Cross-fitted confidence calibration from released validation labels only — source part 2

In [12]:
# [Code Cell]
append_module('calibration.py', r'''    gain=label_report['strict_macro_f1']-global_report['strict_macro_f1']
    policies=fit_policies(scored,gold,labels,fallback)
    policy=policies[1] if gain>=minimum_gain else policies[0]
    return {'policy':policy,'minimum_crossfit_gain':minimum_gain,'label_vs_global_crossfit_gain':gain,
        'global_crossfit_metrics':global_report,'label_crossfit_metrics':label_report,
        'fitted_validation_metrics':metrics(gold,apply_policy(scored,policy),labels),
        'native_validation_metrics':metrics(gold,filter_spans(scored,fallback),labels),
        'scope':'Thresholds use released validation only; checkpoint selection already used validation. No private labels.',
        'refit_policy':'Freeze this policy before training on all released training documents'}
''')


[Markdown Cell]

### config.py

Paths and time/space limits are portable. Changing the number of NER epochs uses a separate committed plan; architecture, optimizer and data settings remain part of the experiment identity.

[Markdown Cell]

Configuration and experiment identity — source part 1

In [13]:
# [Code Cell]
append_module('config.py', r'''# CELL: Configuration and experiment identity
from dataclasses import asdict, dataclass
from pathlib import Path
import json

@dataclass(frozen=True)
class NERConfig:
    """Typed adaptation/head settings; paths and time limits are portable on resume."""
    experiment_id: str = 'E01_A_Full_Finetune'
    family: str = 'a'
    method: str = 'full'
    input_root: str = '/kaggle/input'
    run_dir: str = '/kaggle/working/cs781_phase2/E01_A_Full_Finetune'
    prepared_dir: str | None = None
    parent_run_dir: str | None = None
    tokenizer_path: str | None = None
    metadata_path: str | None = None
    resume_run_dir: str | None = None
    parent_checkpoint: str = 'best'
    teacher_run_dir: str | None = None
    additional_epochs: int = 0
    target_total_epochs: int | None = None
    stop_early: bool = True
    adapter_rank: int = 16
    adapter_alpha: float = 32.0
    bottleneck_width: int = 64
    unfreeze_top_layers: int = 4
    distillation_temperature: float = 2.0
    distillation_weight: float = 0.5
    expected_split_sha256: str = '12de1c79631760f5438b65f3fe8976e98fdfa6a467e506e9f56f3eae420d6408'
    expected_prepared_identity: str = '7ded046ba3e86774047b108b87a8df596006e671f38152f83daa7da4bbdc1960'
    max_epochs: int = 15
    patience: int = 4
    minimum_epochs: int = 3
    backbone_lr: float = 2e-5
    head_lr: float = 3e-4
    weight_decay: float = 0.01
    warmup_fraction: float = 0.10
    min_lr_ratio: float = 0.10
    max_grad_norm: float = 1.0
    micro_batch_size: int = 2
    accumulation_steps: int = 4
    world_size: int = 2
    window_tokens: int = 512  # Includes the initial <s> token.
    window_characters: int = 2048
    overlap_characters: int = 256
    head_width: int = 128
    character_buckets: int = 4096
''')


[Markdown Cell]

Configuration and experiment identity — source part 2

In [14]:
# [Code Cell]
append_module('config.py', r'''    character_width: int = 32
    head_dropout: float = 0.10
    character_context: bool = False
    head_context: str = 'local'
    boundary_loss_weight: float = 0.0
    boundary_weight_max: float = 20.0
    boundary_decoder_weight: float = 0.15
    report_train_f1: bool = True
    activation_checkpointing: bool = False
    class_weight_max: float = 5.0
    confidence_thresholds: tuple[float, ...] = (0.0, 0.50, 0.65, 0.80)
    num_workers: int = 1
    checkpoint_every_steps: int = 500
    max_consecutive_overflows: int = 8
    max_run_seconds: int = 37800
    min_free_bytes: int = 1 << 30
    seed: int = 781
    selection_mode: str = 'heldout'
    fixed_final_epochs: int = 0
    refit_schedule_epochs: int = 15
    fixed_confidence_threshold: float = 0.8
    selection_split_sha256: str = '12de1c79631760f5438b65f3fe8976e98fdfa6a467e506e9f56f3eae420d6408'
    selection_reference_sha256: str = ''

    def validate(self) -> None:
        if type(self.character_context) is not bool:
            raise ValueError('character_context must be a boolean')
        if self.selection_mode not in ('heldout','public_refit'):
            raise ValueError('Use heldout development or explicitly frozen public refit')
        if not 0 <= self.fixed_confidence_threshold <= 1 or self.refit_schedule_epochs < 1:
            raise ValueError('Invalid frozen decoder/schedule recipe')
        if self.selection_mode=='public_refit':
            if self.fixed_final_epochs < 1 or self.additional_epochs or self.target_total_epochs is not None:
                raise ValueError('Public refit requires a fixed fresh epoch recipe; no optimizer extension')
            if self.max_epochs != self.fixed_final_epochs or self.stop_early:
                raise ValueError('Public refit uses its fixed duration without validation early stopping')
        if self.family not in ('a','b','c','d') or self.method not in ('full','lora','adapters','partial','lorma','distillation'):
            raise ValueError('Unknown parent architecture or adaptation method.')
        if (self.method in ('lora','distillation') and self.family!='b'
                or self.method=='adapters' and self.family!='c'
                or self.method=='partial' and self.family!='d'
                or self.method=='lorma' and self.family!='a'):
            raise ValueError('Use the declared six-experiment architecture/method matrix.')
        if self.parent_checkpoint not in ('best','latest'):
            raise ValueError('parent_checkpoint must be best or latest.')
        if type(self.additional_epochs) is not int or self.additional_epochs < 0:
            raise ValueError('additional_epochs must be a nonnegative integer.')
        if self.target_total_epochs is not None and (type(self.target_total_epochs) is not int or self.target_total_epochs < 1):
''')


[Markdown Cell]

Configuration and experiment identity — source part 3

In [15]:
# [Code Cell]
append_module('config.py', r'''            raise ValueError('target_total_epochs must be None or a positive integer.')
        if self.additional_epochs and self.target_total_epochs is not None:
            raise ValueError('Choose additional_epochs OR target_total_epochs, not both.')
        if not 1 <= self.adapter_rank <= 128 or not 0 < self.adapter_alpha <= 256:
            raise ValueError('Invalid low-rank adapter configuration.')
        if not 8 <= self.bottleneck_width <= 256 or not 1 <= self.unfreeze_top_layers <= 12:
            raise ValueError('Invalid bottleneck/partial-training configuration.')
        if not 1 <= self.distillation_temperature <= 10 or not 0 < self.distillation_weight < 1:
            raise ValueError('Invalid knowledge-distillation objective.')
        if not self.experiment_id or '/' in self.experiment_id or '\\' in self.experiment_id:
            raise ValueError('Use a simple experiment_id without path separators.')
        if self.world_size != 2 or min(self.micro_batch_size, self.accumulation_steps) < 1:
            raise ValueError('This experiment requires two GPUs and positive batch sizes.')
        if self.minimum_epochs < 1 or self.max_epochs < 1 or self.patience < 1:
            raise ValueError('Invalid epoch/early-stopping configuration.')
        if not 16 <= self.window_tokens <= 2048 or not 64 <= self.window_characters <= 8192:
            raise ValueError('Invalid token/character window capacity.')
        if not 0 <= self.overlap_characters < self.window_characters // 2:
            raise ValueError('Overlap must be below half the character window.')
        if not 0 < self.backbone_lr <= 1e-3 or not 0 < self.head_lr <= 1e-2:
            raise ValueError('Invalid learning rates.')
        if not 0 <= self.warmup_fraction < 1 or not 0 < self.min_lr_ratio <= 1:
            raise ValueError('Invalid learning-rate schedule.')
        if not 0 <= self.head_dropout < 1 or not 0 <= self.num_workers <= 4:
            raise ValueError('Invalid dropout/workers.')
        if self.head_context not in ('local','token_bigru') or self.head_width % 2:
            raise ValueError('Use local or token_bigru context with an even head width.')
        if not 0 <= self.boundary_loss_weight <= 1:
            raise ValueError('Boundary auxiliary loss weight must be in [0,1].')
        if not 1 <= self.boundary_weight_max <= 100:
            raise ValueError('Invalid train-only boundary class-weight cap.')
        if not 0 <= self.boundary_decoder_weight <= 1:
            raise ValueError('Invalid learned boundary decoding weight.')
        if not 16 <= self.head_width <= 512 or not 256 <= self.character_buckets <= 65536 or not 8 <= self.character_width <= 128:
            raise ValueError('Invalid character-head dimensions.')
        if not 1 <= self.class_weight_max <= 20 or not 0 < self.max_grad_norm <= 10:
            raise ValueError('Invalid class weights or gradient clipping.')
        if self.checkpoint_every_steps != 500:
            raise ValueError('The required checkpoint interval is 500 update boundaries.')
        if not self.confidence_thresholds or any(not 0 <= x <= 1 for x in self.confidence_thresholds):
            raise ValueError('Invalid validation-only confidence threshold grid.')

    def choices(self) -> dict:
        values = asdict(self)
        for key in ('input_root', 'run_dir', 'prepared_dir', 'parent_run_dir', 'tokenizer_path',
                    'metadata_path', 'resume_run_dir', 'max_run_seconds', 'min_free_bytes', 'num_workers',
                    'teacher_run_dir','parent_checkpoint','max_epochs','additional_epochs','target_total_epochs'):
            values.pop(key)
''')


[Markdown Cell]

Configuration and experiment identity — source part 4

In [16]:
# [Code Cell]
append_module('config.py', r'''        return values

    @classmethod
    def load(cls, path: str | Path) -> 'NERConfig':
        values = json.loads(Path(path).read_text(encoding='utf-8'))
        if 'confidence_thresholds' in values:
            values['confidence_thresholds'] = tuple(values['confidence_thresholds'])
        cfg = cls(**values); cfg.validate()
        return cfg
''')


[Markdown Cell]

### data.py

Use unchanged original text. Every training character belongs to exactly one overlapping window’s loss. Character BIO targets represent entity boundaries inside BPE tokens and must reconstruct exact gold spans. No assigned document is sampled away.

[Markdown Cell]

Exact character labels and boundary-preserving windows — source part 1

In [17]:
# [Code Cell]
append_module('data.py', r'''# CELL: Exact character labels and boundary-preserving windows
from collections import OrderedDict
from pathlib import Path
import os
import shutil
import uuid
import numpy as np
import torch
from torch.utils.data import Dataset, Sampler
from tokenizers import Tokenizer
import tokenizers as tokenizers_package
from tqdm.auto import tqdm
from .io_utils import atomic_json, identity, read_json, records, sha256_file

def character_tags(text: str, entities: list, labels: list[str]) -> np.ndarray:
    """Original character string and flat [s,e,label] spans -> BIO tags[len(text)]."""
    tags = np.zeros(len(text), dtype=np.uint8)
    occupied = np.zeros(len(text), dtype=bool)
    for start, end, label in entities:
        if type(start) is not int or type(end) is not int or not 0 <= start < end <= len(text) or label not in labels:
            raise ValueError('Invalid original-character entity bounds/label.')
        if occupied[start:end].any():
            raise ValueError('Overlapping/nested entities require a span head; this flat BIO experiment must not drop them.')
        number = labels.index(label)
        tags[start] = 1 + 2 * number
        tags[start + 1:end] = 2 + 2 * number
        occupied[start:end] = True
    return tags

def character_features(text: str, buckets: int) -> tuple[np.ndarray, np.ndarray]:
    codes = np.fromiter((ord(char) % buckets for char in text), dtype=np.int64, count=len(text)).astype(np.uint16)
    flags = np.fromiter((int(char.isupper()) | (int(char.islower()) << 1) | (int(char.isdigit()) << 2)
                        | (int(char.isspace()) << 3) | (int(not char.isalnum() and not char.isspace()) << 4)
                        | (int(ord(char) > 127) << 5) for char in text), dtype=np.uint8, count=len(text))
    return codes, flags

def token_character_map(text: str, encoding) -> np.ndarray:
    """Assign every ORIGINAL character to a contextual BPE token, including gaps.

    Tokens can cover multiple characters or share offsets for UTF-8 bytes. The
    labels stay on characters; no gold boundary is rounded to a token edge.
    """
    offsets = np.asarray(encoding.offsets, dtype=np.int64)
    if offsets.shape != (len(encoding.ids), 2) or not len(offsets):
        raise ValueError('Nonblank document produced no usable BPE offsets.')
    if (offsets < 0).any() or (offsets > len(text)).any() or (offsets[:, 1] < offsets[:, 0]).any():
        raise ValueError('Tokenizer offsets are outside the original character string.')
    starts = offsets[:, 0]
''')


[Markdown Cell]

Exact character labels and boundary-preserving windows — source part 2

In [18]:
# [Code Cell]
append_module('data.py', r'''    if (starts[1:] < starts[:-1]).any():
        raise ValueError('Nonmonotonic tokenizer offsets; inspect this tokenizer before training.')
    cuts = starts.copy(); cuts[0] = 0
    mapping = np.searchsorted(cuts, np.arange(len(text)), side='right') - 1
    return mapping.clip(0, len(offsets) - 1).astype(np.int32)

def make_windows(mapping: np.ndarray, token_count: int, cfg) -> list[list[int]]:
    """Return [token_start,token_end,char_start,char_end,owner_start,owner_end].

    Context windows overlap. Ownership partitions all original characters once;
    every supervised character is retained, even through large whitespace gaps.
    """
    length, start, covered = len(mapping), 0, 0
    windows = []
    while start < length:
        first = int(mapping[start])
        limit = min(token_count, first + cfg.window_tokens - 1)
        token_boundary = int(np.searchsorted(mapping, limit, side='left'))
        end = min(length, start + cfg.window_characters, token_boundary)
        if end <= start:
            raise ValueError('Token budget cannot represent the current character window.')
        if end <= covered:
            start = covered
            continue
        last = int(mapping[end - 1]) + 1
        windows.append([first, last, start, end, 0, 0]); covered = end
        if end == length:
            break
        overlap = min(cfg.overlap_characters, (end - start) // 4)
        start = max(start + 1, end - overlap)
    if not windows:
        return []
    boundaries = [0] + [(left[3] + right[2]) // 2 for left, right in zip(windows, windows[1:])] + [length]
    for index, window in enumerate(windows):
        window[4:6] = boundaries[index:index + 2]
        assert window[2] <= window[4] < window[5] <= window[3]
        assert window[1] - window[0] + 1 <= cfg.window_tokens
    assert sum(window[5] - window[4] for window in windows) == length
    return windows

''')


[Markdown Cell]

Cache the entire assigned corpus without sampling or normalization — source part 1

In [19]:
# [Code Cell]
append_module('data.py', r'''# CELL: Cache the entire assigned corpus without sampling or normalization
def build_cache(prepared: Path, tokenizer_path: Path, cfg, parent: dict) -> dict:
    from .metrics import decode_tags
    cache_identity = identity({'version': 'character-bio-boundaries-cache-v2', 'split': cfg.expected_split_sha256,
                               'tokenizer': parent['tokenizer_sha256'], 'tokens': cfg.window_tokens,
                               'characters': cfg.window_characters, 'overlap': cfg.overlap_characters,
                               'buckets': cfg.character_buckets, 'class_weight_max': cfg.class_weight_max,
                               'boundary_weight_max':cfg.boundary_weight_max,
                               'tokenizers_version':tokenizers_package.__version__, 'numpy_version':np.__version__})
    root = Path(cfg.run_dir) / 'cache' / cache_identity[:12]
    if (root / 'cache_manifest.json').is_file():
        manifest = read_json(root / 'cache_manifest.json')
        if manifest['identity'] != cache_identity:
            raise ValueError('Cache identity mismatch.')
        for name, receipt in manifest['artifacts'].items():
            path = root / name
            if path.stat().st_size != receipt['bytes'] or sha256_file(path) != receipt['sha256']:
                raise ValueError(f'Corrupted NER cache: {name}')
        return {'directory': str(root), **manifest}
    stage = root.with_name(root.name + '.staging_' + uuid.uuid4().hex); stage.mkdir(parents=True)
    tokenizer = Tokenizer.from_file(str(tokenizer_path)); tokenizer.no_padding(); tokenizer.no_truncation()
    labels = read_json(prepared / 'prepared_manifest.json')['labels']
    counts = np.zeros(25, dtype=np.int64); documents, index_tables = {}, {}
    for split in ('train', 'validation', 'test'):
        documents[split], index_tables[split] = [], []
        output = stage / split; output.mkdir()
        for doc_number, row in enumerate(tqdm(records(prepared / (split + '.jsonl')), desc=f'Cache all {split} documents')):
            text = row['text']
            if not isinstance(text, str):
                raise ValueError('Prepared text must be an unchanged string.')
            entry = {'id': row['id'], 'characters': len(text), 'directory': f'{split}/{doc_number}',
                     'blank': not text.strip()}
            documents[split].append(entry)
            if entry['blank']:
                if split != 'test':
                    raise ValueError('Blank training/validation document.')
                continue  # The ID remains in documents and receives [] at prediction.
            encoding = tokenizer.encode(text, add_special_tokens=False)
            mapping = token_character_map(text, encoding)
            codes, flags = character_features(text, cfg.character_buckets)
            tags = character_tags(text, row['entities'], labels) if split != 'test' else np.zeros(len(text), dtype=np.uint8)
            ends=np.zeros(len(text),dtype=np.uint8)
            if split!='test':
                for start,end,label in row['entities']:
                    ends[end-1]=labels.index(label)+1  # End-exclusive gold -> final entity character.
            if split != 'test':
                reconstructed = decode_tags(tags, labels)
                if sorted(reconstructed) != sorted([list(entity[:3]) for entity in row['entities']]):
''')


[Markdown Cell]

Cache the entire assigned corpus without sampling or normalization — source part 2

In [20]:
# [Code Cell]
append_module('data.py', r'''                    raise ValueError('Gold-to-character BIO round trip changed an exact entity boundary.')
            if split == 'train':
                counts += np.bincount(tags, minlength=25)
            folder = stage / entry['directory']; folder.mkdir()
            for name, value in {'tokens': np.asarray(encoding.ids, dtype=np.int32), 'mapping': mapping,
                                'characters': codes, 'flags': flags, 'tags': tags, 'ends':ends}.items():
                np.save(folder / (name + '.npy'), value, allow_pickle=False)
            windows = make_windows(mapping, len(encoding.ids), cfg)
            index_tables[split].extend([[doc_number, *window] for window in windows])
        table = np.asarray(index_tables[split], dtype=np.int64).reshape(-1, 7)
        np.save(stage / (split + '_windows.npy'), table, allow_pickle=False)
    # All training characters contribute. Weights use no validation/test labels.
    weights = np.sqrt(counts.sum() / (25 * counts.clip(1))).clip(0.25, cfg.class_weight_max).astype(np.float32)
    weights /= weights.mean()
    np.save(stage / 'class_weights.npy', weights, allow_pickle=False)
    boundary_counts=np.concatenate(([counts.sum()-counts[1::2].sum()],counts[1::2]))
    boundary_weights=np.sqrt(counts.sum()/(13*boundary_counts.clip(1))).clip(.25,cfg.boundary_weight_max).astype(np.float32)
    boundary_weights/=boundary_weights.mean()
    np.save(stage/'boundary_class_weights.npy',boundary_weights,allow_pickle=False)
    normalizers, boundary_normalizers = [], []
    for doc, _, _, _, _, owner_start, owner_end in index_tables['train']:
        tags = np.load(stage / documents['train'][doc]['directory'] / 'tags.npy', mmap_mode='r')
        normalizers.append(float(weights[tags[owner_start:owner_end]].sum(dtype=np.float64)))
        owned=tags[owner_start:owner_end]
        starts=np.where(owned%2==1,(owned+1)//2,0)
        ends=np.load(stage/documents['train'][doc]['directory']/'ends.npy',mmap_mode='r')
        boundary_normalizers.append([float(boundary_weights[starts].sum(dtype=np.float64)),
            float(boundary_weights[ends[owner_start:owner_end]].sum(dtype=np.float64))])
        del ends
        del tags,owned,starts  # Release all mapping views before atomic publication.
    np.save(stage / 'window_weights.npy', np.asarray(normalizers, dtype=np.float64), allow_pickle=False)
    np.save(stage/'window_boundary_weights.npy',np.asarray(boundary_normalizers,dtype=np.float64),allow_pickle=False)
    atomic_json(stage / 'documents.json', documents)
    report = {'identity': cache_identity, 'labels': labels, 'documents': documents,
              'dependencies': {'tokenizers':tokenizers_package.__version__, 'numpy':np.__version__},
              'window_counts': {key: len(value) for key, value in index_tables.items()},
              'training_character_counts': counts.tolist(), 'class_weights': weights.tolist(),
              'boundary_character_counts':boundary_counts.tolist(),'boundary_class_weights':boundary_weights.tolist(),
              'all_assigned_documents_used': True, 'every_training_character_owned_once': True,
              'gold_character_roundtrip': 'passed', 'text_normalization': 'none',
              'head': 'Learned character BIO, preserving boundaries inside BPE tokens'}
    artifacts = {str(path.relative_to(stage)).replace('\\','/'): {'bytes': path.stat().st_size, 'sha256': sha256_file(path)}
                 for path in stage.rglob('*') if path.is_file()}
    manifest = {**report, 'artifacts': artifacts}
    atomic_json(stage / 'cache_manifest.json', manifest)
    if root.exists():
        raise ValueError('Uncommitted cache already exists; choose a fresh run directory.')
    os.replace(stage, root)
''')


[Markdown Cell]

Cache the entire assigned corpus without sampling or normalization — source part 3

In [21]:
# [Code Cell]
append_module('data.py', r'''    return {'directory': str(root), **manifest}

''')


[Markdown Cell]

Memory-mapped deterministic windows and dynamic padding — source part 1

In [22]:
# [Code Cell]
append_module('data.py', r'''# CELL: Memory-mapped deterministic windows and dynamic padding
class WindowDataset(Dataset):
    """Index -> token IDs[T], character inputs[C], BIO targets[C] and provenance."""
    def __init__(self, cache_dir: str, split: str, bos_id: int, teacher_dir: str | None=None):
        self.root, self.split, self.bos_id = Path(cache_dir), split, bos_id
        self.documents = read_json(self.root / 'documents.json')[split]
        self.windows = np.load(self.root / (split + '_windows.npy'), mmap_mode='r')
        self.opened = OrderedDict()
        self.teacher_dir=Path(teacher_dir) if teacher_dir and split=='train' else None

    def __len__(self) -> int:
        return len(self.windows)

    def close(self) -> None:
        for values in getattr(self,'opened',{}).values():
            for array in values.values():
                array._mmap.close()
        if hasattr(self,'opened'):
            self.opened.clear()
        if hasattr(self,'windows'):
            self.windows._mmap.close()

    def arrays(self, doc: int) -> dict:
        if doc not in self.opened:
            folder = self.root / self.documents[doc]['directory']
            self.opened[doc] = {name: np.load(folder / (name + '.npy'), mmap_mode='r')
                                for name in ('tokens','mapping','characters','flags','tags','ends')}
            if self.teacher_dir:
                self.opened[doc]['teacher']=np.load(self.teacher_dir/(str(doc)+'.npy'),mmap_mode='r')
            if len(self.opened) > 4:
                self.opened.popitem(last=False)
        self.opened.move_to_end(doc)
        return self.opened[doc]

    def __getitem__(self, index: int) -> dict:
        dummy = index == -1
        doc, ts, te, start, end, own_start, own_end = map(int, self.windows[0 if dummy else index])
        arrays = self.arrays(doc)
        targets = np.array(arrays['tags'][start:end], dtype=np.int64)
        targets[:own_start - start] = -100; targets[own_end - start:] = -100
        if dummy:
            targets[:] = -100
        result={'input_ids': torch.tensor([self.bos_id, *arrays['tokens'][ts:te]], dtype=torch.long),
                'character_ids': torch.tensor(np.array(arrays['characters'][start:end]), dtype=torch.long),
                'flags': torch.tensor(np.array(arrays['flags'][start:end]), dtype=torch.long),
                'token_positions': torch.tensor(np.array(arrays['mapping'][start:end]) - ts + 1, dtype=torch.long),
                'targets': torch.from_numpy(targets), 'doc': doc, 'start': start, 'end': end}
        start_targets=np.where(targets%2==1,(targets+1)//2,0)
''')


[Markdown Cell]

Memory-mapped deterministic windows and dynamic padding — source part 2

In [23]:
# [Code Cell]
append_module('data.py', r'''        end_targets=np.array(arrays['ends'][start:end],dtype=np.int64)
        start_targets[targets==-100]=-100; end_targets[targets==-100]=-100
        result.update(start_targets=torch.from_numpy(start_targets),end_targets=torch.from_numpy(end_targets))
        if self.teacher_dir:
            result['teacher_logits']=torch.tensor(np.array(arrays['teacher'][start:end]),dtype=torch.float32)
        return result

def collate_windows(rows: list[dict]) -> dict:
    result = {}
    token_length = max(len(row['input_ids']) for row in rows)
    character_length = max(len(row['targets']) for row in rows)
    for key in ('input_ids','character_ids','flags','token_positions','targets','start_targets','end_targets'):
        length = token_length if key == 'input_ids' else character_length
        value = torch.full((len(rows), length), -100 if key.endswith('targets') else 0, dtype=torch.long)
        for index, row in enumerate(rows):
            value[index, :len(row[key])] = row[key]
        result[key] = value
    result['attention_mask'] = torch.arange(token_length)[None] < torch.tensor([len(row['input_ids']) for row in rows])[:,None]
    result['character_mask'] = torch.arange(character_length)[None] < torch.tensor([len(row['targets']) for row in rows])[:,None]
    result['metadata'] = [{'doc': row['doc'], 'start': row['start'], 'end': row['end']} for row in rows]
    if 'teacher_logits' in rows[0]:
        teacher=torch.zeros((len(rows),character_length,25),dtype=torch.float32)
        for index,row in enumerate(rows):
            teacher[index,:len(row['targets'])]=row['teacher_logits']
        result['teacher_logits']=teacher
    return result

''')


[Markdown Cell]

Exhaustive distributed batch planning without dropped tails — source part 1

In [24]:
# [Code Cell]
append_module('data.py', r'''# CELL: Exhaustive distributed batch planning without dropped tails
class RankBatchSampler(Sampler):
    """Every global window once/epoch; -1 creates an unsupervised synchronization row."""
    def __init__(self, size: int, cfg, rank: int, epoch: int, start_batch: int = 0):
        self.cfg, self.rank, self.start_batch = cfg, rank, start_batch
        self.order = np.random.default_rng(cfg.seed + epoch).permutation(size)
        self.global_batch = cfg.world_size * cfg.micro_batch_size * cfg.accumulation_steps
        self.steps = (size + self.global_batch - 1) // self.global_batch

    def global_indices(self, step: int) -> np.ndarray:
        return self.order[step*self.global_batch:(step+1)*self.global_batch]

    def __iter__(self):
        width = self.cfg.micro_batch_size * self.cfg.world_size
        for step in range(self.start_batch, self.steps):
            indices = self.global_indices(step)
            for micro in range(self.cfg.accumulation_steps):
                first = micro*width + self.rank*self.cfg.micro_batch_size
                local = indices[first:first+self.cfg.micro_batch_size].tolist()
                yield local or [-1]

    def __len__(self) -> int:
        return (self.steps - self.start_batch) * self.cfg.accumulation_steps
''')


[Markdown Cell]

### distill.py

A larger completed NER teacher supplies fixed character-aligned log probabilities for training only. Cache identities include teacher weights and the exact split. Subsequent student resumes retain this same teacher cache.

[Markdown Cell]

Character-aligned teacher probabilities with no validation/test training — source part 1

In [25]:
# [Code Cell]
append_module('distill.py', r'''# CELL: Character-aligned teacher probabilities with no validation/test training
from dataclasses import replace
import json
import os
from pathlib import Path
import shutil
import uuid
import numpy as np
import torch
from torch.nn import functional as F
from torch.utils.data import DataLoader,Subset
from tqdm.auto import tqdm
from .config import NERConfig
from .data import WindowDataset,build_cache,collate_windows
from .io_utils import atomic_json,ensure_space,identity,read_json,sha256_file
from .model import build_model,forward_batch
from .parents import source_hashes
from .state import Checkpoints

LEGACY_SOURCE_SHA256={'__init__.py': 'b1f4a751377e8d47e073ebfe55b1f604470a7b1102dedd47feaa5dd29f13754f', 'adaptation.py': 'b5f452257274cf43e41eb0c489807cf8e643977419584b301245d469d4a0e2ed', 'artifacts.py': 'ad515bd2ee897afb9c8a98c90b14b422fcb751802837e50929b8f667aaf24b2c', 'config.py': 'a209e1d563ec692a12dc6bcce9d36b1950c0afaa310a959668ec9c76e0e2188a', 'data.py': '727aca69cfb16bf41d6752bbfee644f57d05855bceda536aaaa7b75195a1be7e', 'distill.py': '4fbf420829c752f3ad363e441c5c2dd695d62c9ab996c8c0d69a08cd6b754412', 'io_utils.py': '3f9d828342c819690d390f0280c01f9eda160aab59c9cf424d6b3c2a31813e73', 'metrics.py': '20f85e410594dc2b3e2ce1aec58f0fb77959e7d90e587b2b1307f84c3e43ee21', 'model.py': '7aacf41547e87f8bd6a0420ce590d7b9d3d520d981c650d53d436bc94ddc1ad9', 'parents.py': '834710b94c3f82eb18b48abf752727e30d245ebbe58f55408263b54fdddcd660', 'predict.py': '4ad3d27e32d79bda7b29ba35a242361732b591278929470691ee7cdfbba0ca01', 'prepare.py': '9b64657d6e9aa1b6c22f1a2e533fe6a122eb784ba3b1b844f75d701f39725075', 'state.py': 'f05cc134dcbff7b59050460dccc6d61cbd5b75ad6920175526b459ef3a1eeebf', 'train.py': '03f80aca08975ab125c3f0e2ec31a371c033ce3527427f25949cdead3e421c57'}

EXPERIMENT_SOURCE_SHA256={'adaptation.py': 'b5f452257274cf43e41eb0c489807cf8e643977419584b301245d469d4a0e2ed', 'artifacts.py': 'ad515bd2ee897afb9c8a98c90b14b422fcb751802837e50929b8f667aaf24b2c', 'config.py': 'b1f8d2f7d9a4584e376802de5deca5aadaa40ece408a42584267feeda8939ec8', 'data.py': 'd7b6fabbf6f24395f35b5f7a7e2b790401b9d34a33a6a0e0fe8fa0976e6e6332', 'distill.py': 'd6e58a9c340e9f4ea8b8d0acf61a7585be94fcf9fa45e103c5ecc0c29dd159fe', 'io_utils.py': '3f9d828342c819690d390f0280c01f9eda160aab59c9cf424d6b3c2a31813e73', 'metrics.py': 'e5e611b0c176543531ddbd97b9f3b6f0bd4d34b3b27d430a71befb2fdabd1dbb', 'model.py': 'd1612ed0700a2fe83a7c4e4b66ded119dedb518bb30763f21695057225773ed8', 'parents.py': '834710b94c3f82eb18b48abf752727e30d245ebbe58f55408263b54fdddcd660', 'predict.py': 'f85aeb9f056e3e1ed3aaad89a1a74337d17a40b4a75e1c1a73e1428dc306f8cd', 'prepare.py': '77ad5c5add32b9902a70348194550e4e67c110dce91ef2a8be5e8140637667fa', 'search.py': '0650caca0cdf6ab6eded43105221c4715cb2b69851a3f5a53283259cff5cd647', 'state.py': 'f05cc134dcbff7b59050460dccc6d61cbd5b75ad6920175526b459ef3a1eeebf', 'train.py': 'b995036cd5acb4552aa8d184e2ada6c85ddff8e76dc1930a7301dcddbeedee12', '__init__.py': 'a6ee8e23898c6dd802a7c3285bd4c9cef008152a8dbac95b7ddbd2219b42876f'}

FINAL_SOURCE_SHA256={'adaptation.py': 'b5f452257274cf43e41eb0c489807cf8e643977419584b301245d469d4a0e2ed', 'artifacts.py': 'ad515bd2ee897afb9c8a98c90b14b422fcb751802837e50929b8f667aaf24b2c', 'calibration.py': 'fe111bb74274908034d3cb9d161cd35934f36c83fe80745dc42bd2a55fa5112d', 'config.py': '611787e4abb39cd46be389c462fdb6f91bfb04090b2cff7bf98dc84a13a1de7f', 'data.py': 'd7b6fabbf6f24395f35b5f7a7e2b790401b9d34a33a6a0e0fe8fa0976e6e6332', 'distill.py': '1c0f351175cf8fbf15f8ed83edfc47f099c5ed2ace0e7f2ebedb751bb669740d', 'io_utils.py': '3f9d828342c819690d390f0280c01f9eda160aab59c9cf424d6b3c2a31813e73', 'metrics.py': 'e5e611b0c176543531ddbd97b9f3b6f0bd4d34b3b27d430a71befb2fdabd1dbb', 'model.py': 'd1612ed0700a2fe83a7c4e4b66ded119dedb518bb30763f21695057225773ed8', 'parents.py': '834710b94c3f82eb18b48abf752727e30d245ebbe58f55408263b54fdddcd660', 'predict.py': 'd519ca116eb43342102ed7982dd08f8f707667ac7d4f80ebbf0a26d9c82fddc7', 'prepare.py': 'd79835f15988209db2c40dfeda861a3e4e34aa190342405f64d59189c3024ae3', 'public_data.py': 'a6eec9fb24e66d8f5280e4618e0ce6214e525cbf48751d9b30c2b9a4bd56f9be', 'state.py': 'f05cc134dcbff7b59050460dccc6d61cbd5b75ad6920175526b459ef3a1eeebf', 'train.py': 'c0cbb46c8866bb13a175a6b4e5b10aee0a8f45a163eea86f100ed7195bcd0885', 'workflow.py': 'd725cb6562a957ac279d9b1a1b6e6aec65b87ea9200883e3d3f1fa637290ac07', '__init__.py': 'a6ee8e23898c6dd802a7c3285bd4c9cef008152a8dbac95b7ddbd2219b42876f'}

def distillation_loss(student,teacher_logp,targets,weights,temperature: float):
    """Weighted sum T^2 KL(teacher || student) on owned training characters only."""
    mask=targets!=-100
    teacher=F.softmax(teacher_logp.float()/temperature,dim=-1)
    log_student=F.log_softmax(student.float()/temperature,dim=-1)
    kl=F.kl_div(log_student,teacher,reduction='none').sum(-1)*(temperature**2)
    effective=weights[targets.clamp_min(0)]*mask
    return (kl*effective).sum()

def relocated_teacher(root: Path,value: dict,cfg,catalog: dict) -> tuple[dict,NERConfig]:
    """Resolve a saved NER output independently of its former /working paths."""
    teacher_cfg=NERConfig(**{**value['config'],'confidence_thresholds':tuple(value['config']['confidence_thresholds'])})
    if teacher_cfg.family not in ('a','c','d') or value.get('version') not in ('cs781-flexible-ner-v1','cs781-ner-experiments-v2','cs781-ner-final-v1','cs781-ner-repair-v2'):
        raise ValueError('Teacher must be a completed A/C/D NER experiment from these flexible notebooks')
    if teacher_cfg.selection_mode!='heldout' or teacher_cfg.expected_split_sha256!=cfg.selection_split_sha256 or value['labels']!=read_json(Path(cfg.prepared_dir)/'prepared_manifest.json')['labels']:
        raise ValueError('Teacher and student must use identical document split and BIO label order')
    if value['environment']['torch']!=str(torch.__version__):
        raise ValueError('Use the teacher recorded PyTorch build '+value['environment']['torch'])
    legacy=value['version']=='cs781-flexible-ner-v1'
    package={'cs781-flexible-ner-v1':'ner_flexible','cs781-ner-experiments-v2':'ner_experiments','cs781-ner-final-v1':'ner_final','cs781-ner-repair-v2':'ner_repair'}[value['version']]
    expected={'cs781-flexible-ner-v1':LEGACY_SOURCE_SHA256,'cs781-ner-experiments-v2':EXPERIMENT_SOURCE_SHA256,'cs781-ner-final-v1':FINAL_SOURCE_SHA256}.get(value['version'],source_hashes(Path(__file__).parent))
    if source_hashes(root/'code'/package)!=value['runtime_source_sha256'] or expected!=value['runtime_source_sha256']:
        raise ValueError('Teacher runtime changed/incomplete or not a verified supported version')
''')


[Markdown Cell]

Character-aligned teacher probabilities with no validation/test training — source part 2

In [26]:
# [Code Cell]
append_module('distill.py', r'''    if legacy and (teacher_cfg.head_context!='local' or teacher_cfg.boundary_loss_weight):
        raise ValueError('Legacy teacher must have the original local BIO head')
    job=dict(value); job['prepared_dir']=cfg.prepared_dir
    job['parent_source_dir']=str(root/'parent_source'/f'model_{teacher_cfg.family}')
    job['tokenizer_path']=str(root/'tokenizer/tokenizer.json'); job['parent_contract']=catalog[teacher_cfg.family]
    if sha256_file(job['tokenizer_path'])!=job['parent']['tokenizer_sha256']:
        raise ValueError('Teacher tokenizer changed/missing')
    return job,replace(teacher_cfg,run_dir=str(Path(cfg.run_dir)/'teacher_workspace'),prepared_dir=cfg.prepared_dir,expected_split_sha256=cfg.expected_split_sha256,expected_prepared_identity=cfg.expected_prepared_identity)

@torch.inference_mode()
def teacher_document_scores(model,job,cfg,device):
    """Yield one full train document's fused log probabilities at a time."""
    model.eval(); dataset=WindowDataset(job['cache']['directory'],'train',1)
    try:
        for doc,entry in enumerate(tqdm(dataset.documents,desc='Teacher: all assigned training documents')):
            scores=np.zeros((entry['characters'],25),dtype=np.float32)
            total=np.zeros(entry['characters'],dtype=np.float32)
            indices=np.flatnonzero(dataset.windows[:,0]==doc).tolist()
            loader=DataLoader(Subset(dataset,indices),batch_size=cfg.micro_batch_size,collate_fn=collate_windows,num_workers=0)
            for batch in loader:
                with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=device.type=='cuda'):
                    logits=forward_batch(model,batch,device)
                logp=logits.float().log_softmax(-1).cpu().numpy()
                if not np.isfinite(logp).all():
                    raise ValueError('Nonfinite teacher predictions')
                for index,meta in enumerate(batch['metadata']):
                    length=meta['end']-meta['start']; positions=np.arange(length)
                    weight=np.minimum(positions+1,length-positions).clip(1,128).astype(np.float32)
                    section=slice(meta['start'],meta['end'])
                    scores[section]+=logp[index,:length]*weight[:,None]; total[section]+=weight
            if (total<=0).any():
                raise ValueError('Teacher failed to cover the full training document')
            yield doc,entry,scores/total[:,None]
    finally:
        dataset.close()

''')


[Markdown Cell]

Choose the strongest validated larger teacher and cache it once — source part 1

In [27]:
# [Code Cell]
append_module('distill.py', r'''# CELL: Choose the strongest validated larger teacher and cache it once
def prepare_teacher_cache(student_job,cfg,catalog,previous: dict | None=None) -> dict:
    root=Path(cfg.run_dir)/'teacher_cache'
    if previous:
        source=Path(previous['directory']); destination=Path(cfg.run_dir)/previous['relative_directory']
        manifest=read_json(source/'teacher_manifest.json')
        if manifest['identity']!=previous['identity']:
            raise ValueError('Saved distillation cache identity changed')
        for name,receipt in manifest['artifacts'].items():
            path=source/name
            if path.stat().st_size!=receipt['bytes'] or sha256_file(path)!=receipt['sha256']:
                raise ValueError('Saved distillation cache corrupted/incomplete')
        if source.resolve()!=destination.resolve():
            ensure_space(Path(cfg.run_dir),sum(row['bytes'] for row in manifest['artifacts'].values()),cfg.min_free_bytes)
            if destination.exists():
                raise ValueError('Uncommitted teacher cache already exists; use a fresh run directory')
            destination.parent.mkdir(parents=True,exist_ok=True); shutil.copytree(source,destination)
        return {**previous,'directory':str(destination)}
    prepared=Path(student_job['prepared_dir']); search=[Path(cfg.teacher_run_dir)/'job.json'] if cfg.teacher_run_dir else sorted(Path(cfg.input_root).rglob('job.json'))
    choices=[]
    for path in search:
        value=read_json(path)
        if value.get('version') not in ('cs781-flexible-ner-v1','cs781-ner-experiments-v2','cs781-ner-final-v1','cs781-ner-repair-v2') or value.get('config',{}).get('family') not in ('a','c','d'):
            continue
        summary_path=path.parent/'training_summary.json'
        if value.get('config',{}).get('selection_mode','heldout')!='heldout' or not summary_path.is_file() or read_json(summary_path).get('status')!='complete':
            continue
        teacher_job,teacher_cfg=relocated_teacher(path.parent,value,replace(cfg,prepared_dir=str(prepared)),catalog)
        manager=Checkpoints(path.parent,value['fingerprint'],cfg.min_free_bytes)
        best=manager.best()
        if best:
            choices.append((best[1]['validation_macro_f1'],path.parent,teacher_job,teacher_cfg,best))
    if not choices:
        raise FileNotFoundError('Attach a completed larger Phase 2 A/C/D NER output with best.pt/receipt, code, job.json, tokenizer and parent_source. A Phase 1 checkpoint is not an NER teacher.')
    score,teacher_root,teacher_job,teacher_cfg,best=max(choices,key=lambda row:(row[0],row[4][1]['step']))
    state=torch.load(best[0],map_location='cpu',weights_only=True,mmap=True)
    if state['fingerprint']!=teacher_job['fingerprint'] or state['step']!=best[1]['step']:
        raise ValueError('Teacher checkpoint/receipt identity disagreement')
    digest=identity({'teacher_sha256':best[1]['sha256'],'teacher_fingerprint':teacher_job['fingerprint'],
                     'student_split':cfg.expected_split_sha256,'student_cache':student_job['cache']['identity'],
                     'labels':student_job['labels'],'fusion':'triangular-log-probability-v1'})
    destination=root/digest[:12]
    if destination.exists():
        prior={'identity':digest,'directory':str(destination),'relative_directory':str(destination.relative_to(Path(cfg.run_dir))),
               'teacher_sha256':best[1]['sha256'],'teacher_validation_macro_f1':score}
        return prepare_teacher_cache(student_job,cfg,catalog,prior)
    teacher_job['cache']=build_cache(prepared,Path(teacher_job['tokenizer_path']),teacher_cfg,teacher_job['parent'])
    device=torch.device('cuda',0); model=build_model(teacher_job,teacher_cfg,device,load_parent=False)
''')


[Markdown Cell]

Choose the strongest validated larger teacher and cache it once — source part 2

In [28]:
# [Code Cell]
append_module('distill.py', r'''    model.load_state_dict(state['model'],strict=True); del state
    student_meta=build_model(student_job,cfg,torch.device('meta'),load_parent=False)
    student_count=student_meta.parameter_report['active_ner_parameters']; del student_meta
    teacher_count=model.parameter_report['active_ner_parameters']
    if teacher_count<=student_count:
        raise ValueError('Teacher must be larger than the active B NER student')
    estimate=sum(entry['characters'] for entry in student_job['cache']['documents']['train'])*25*2
    ensure_space(Path(cfg.run_dir),estimate,cfg.min_free_bytes)
    stage=destination.with_name(destination.name+'.staging_'+uuid.uuid4().hex); stage.mkdir(parents=True)
    for doc,entry,scores in teacher_document_scores(model,teacher_job,teacher_cfg,device):
        expected=student_job['cache']['documents']['train'][doc]
        if entry['id']!=expected['id'] or entry['characters']!=expected['characters']:
            raise ValueError('Teacher/student original character alignment differs')
        np.save(stage/(str(doc)+'.npy'),scores.astype(np.float16),allow_pickle=False)
    del model; torch.cuda.empty_cache()
    provenance={'identity':digest,'teacher_sha256':best[1]['sha256'],'teacher_experiment_id':teacher_cfg.experiment_id,
                'teacher_validation_macro_f1':score,'teacher_active_parameters':teacher_count,'student_active_parameters':student_count,
                'teacher_selection':'highest saved validation strict Macro-F1 among attached larger A/C/D teachers',
                'training_documents':len(student_job['cache']['documents']['train']),'validation_test_used_for_student_loss':False,
                'alignment':'unchanged original document character offsets, independently of tokenizer vocabulary'}
    receipts={p.name:{'bytes':p.stat().st_size,'sha256':sha256_file(p)} for p in stage.glob('*.npy')}
    atomic_json(stage/'teacher_manifest.json',{**provenance,'artifacts':receipts})
    os.replace(stage,destination)
    return {**provenance,'directory':str(destination),'relative_directory':str(destination.relative_to(Path(cfg.run_dir)))}
''')


[Markdown Cell]

### io_utils.py

Atomic files, checksums, exclusive ownership and disk reservations prevent mixed or incomplete runs.

[Markdown Cell]

Strict JSON, checksums and atomic writes — source part 1

In [29]:
# [Code Cell]
append_module('io_utils.py', r'''# CELL: Strict JSON, checksums and atomic writes
from contextlib import contextmanager
import hashlib
import json
import os
from pathlib import Path
import shutil
import uuid

def strict_pairs(pairs: list[tuple[str, object]]) -> dict:
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f'Duplicate JSON key: {key}')
        result[key] = value
    return result

def reject_constant(value: str):
    raise ValueError(f'Nonfinite JSON value: {value}')

def read_json(path: str | Path) -> dict:
    return json.loads(Path(path).read_text(encoding='utf-8'), object_pairs_hook=strict_pairs,
                      parse_constant=reject_constant)

def records(path: str | Path):
    with Path(path).open(encoding='utf-8') as stream:
        for line_number, line in enumerate(stream, 1):
            if line.strip():
                yield json.loads(line, object_pairs_hook=strict_pairs, parse_constant=reject_constant)

def sha256_file(path: str | Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 << 20), b''):
            digest.update(chunk)
    return digest.hexdigest()

def identity(value: dict) -> str:
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(',', ':'),
                                     allow_nan=False).encode()).hexdigest()

def atomic_json(path: str | Path, value: dict) -> None:
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.partial')
    with temporary.open('w', encoding='utf-8', newline='\n') as stream:
        json.dump(value, stream, ensure_ascii=False, indent=2, allow_nan=False)
        stream.write('\n'); stream.flush(); os.fsync(stream.fileno())
    os.replace(temporary, path)
''')


[Markdown Cell]

Strict JSON, checksums and atomic writes — source part 2

In [30]:
# [Code Cell]
append_module('io_utils.py', r'''
def ensure_space(directory: Path, needed: int, reserve: int) -> None:
    if shutil.disk_usage(directory).free < needed + reserve:
        raise OSError('Insufficient working disk. Start a fresh Kaggle session and attach the saved output; do not delete committed checkpoints.')

''')


[Markdown Cell]

Exclusive ownership and verified checkpoint copies — source part 1

In [31]:
# [Code Cell]
append_module('io_utils.py', r'''# CELL: Exclusive ownership and verified checkpoint copies
@contextmanager
def run_lock(directory: Path):
    """Linux advisory lock released by the OS after a killed Kaggle process."""
    import fcntl
    directory.mkdir(parents=True, exist_ok=True)
    with (directory / '.training.lock').open('a+') as stream:
        try:
            fcntl.flock(stream.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError as error:
            raise RuntimeError('This experiment is already running. Do not launch a second training cell.') from error
        try:
            yield
        finally:
            fcntl.flock(stream.fileno(), fcntl.LOCK_UN)

def publish_copy(source: Path, destination: Path) -> None:
    """Publish a source without modifying it; hard-link only when supported."""
    if source.resolve() == destination.resolve():
        return
    temporary = destination.with_name(destination.name + '.' + uuid.uuid4().hex + '.partial')
    try:
        os.link(source, temporary)
    except OSError:
        shutil.copyfile(source, temporary)
    with temporary.open('r+b') as stream:
        os.fsync(stream.fileno())
    os.replace(temporary, destination)
''')


[Markdown Cell]

### metrics.py

Exact document/start/end/label matches define TP. $\mathrm{MacroF1}=\frac1{12}\sum_c\frac{2TP_c}{2TP_c+FP_c+FN_c}$. Decode legal BIO transitions and select confidence on validation only.

[Markdown Cell]

Exact BIO decoding and linear-time constrained Viterbi — source part 1

In [32]:
# [Code Cell]
append_module('metrics.py', r'''# CELL: Exact BIO decoding and linear-time constrained Viterbi
import numpy as np

def decode_tags(tags, labels: list[str]) -> list[list]:
    """Character BIO IDs[C] -> exact original [start,end,label] triples."""
    result, start, active = [], None, None
    for index, value in enumerate(tags):
        tag = int(value)
        label = labels[(tag - 1) // 2] if tag else None
        beginning = bool(tag % 2)
        if active is not None and (tag == 0 or beginning or label != active):
            result.append([start, index, active]); start, active = None, None
        if tag and active is None:
            start, active = index, label
    if active is not None:
        result.append([start, len(tags), active])
    return result

def constrained_viterbi(log_probabilities: np.ndarray) -> np.ndarray:
    """BIO transitions: I(label) follows only B/I of the same label; O/B follow any."""
    if not len(log_probabilities):
        return np.zeros(0, dtype=np.uint8)
    if log_probabilities.shape[1] != 25 or not np.isfinite(log_probabilities).all():
        raise ValueError('Invalid character log probabilities.')
    back = np.zeros((len(log_probabilities), 25), dtype=np.uint8)
    score = log_probabilities[0].copy(); score[2::2] = -np.inf
    all_new = np.array([0, *range(1,25,2)])
    begin_ids, inside_ids = np.arange(1,25,2), np.arange(2,25,2)
    for index in range(1, len(log_probabilities)):
        best = int(score.argmax())
        previous_inside = np.where(score[begin_ids] >= score[inside_ids], begin_ids, inside_ids)
        new = np.empty_like(score)
        new[all_new] = score[best] + log_probabilities[index,all_new]
        new[inside_ids] = score[previous_inside] + log_probabilities[index,inside_ids]
        back[index,all_new] = best; back[index,inside_ids] = previous_inside
        score = new
    tags = np.zeros(len(log_probabilities), dtype=np.uint8); tags[-1] = score.argmax()
    for index in range(len(tags)-1,0,-1):
        tags[index-1] = back[index,tags[index]]
    return tags

''')


[Markdown Cell]

Strict metrics and model-confidence filtering — source part 1

In [33]:
# [Code Cell]
append_module('metrics.py', r'''# CELL: Strict metrics and model-confidence filtering
def metrics(gold: dict, predictions: dict, labels: list[str]) -> dict:
    if set(gold) != set(predictions):
        raise ValueError('Held-out prediction IDs differ from gold IDs.')
    per_label = {}
    for label in labels:
        tp = fp = fn = 0
        for key in gold:
            g = {tuple(row[:3]) for row in gold[key] if row[2] == label}
            p = {tuple(row[:3]) for row in predictions[key] if row[2] == label}
            tp += len(g & p); fp += len(p - g); fn += len(g - p)
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0
        per_label[label] = dict(tp=tp, fp=fp, fn=fn, precision=precision, recall=recall, f1=f1)
    return {'strict_macro_f1': sum(row['f1'] for row in per_label.values()) / 12,
            'strict_macro_precision': sum(row['precision'] for row in per_label.values()) / 12,
            'strict_macro_recall': sum(row['recall'] for row in per_label.values()) / 12,
            'per_label': per_label, 'examples': len(gold)}

''')


[Markdown Cell]

Learned boundary evidence with the same legal BIO grammar — source part 1

In [34]:
# [Code Cell]
append_module('metrics.py', r'''# CELL: Learned boundary evidence with the same legal BIO grammar
def boundary_viterbi(bio_logp: np.ndarray,start_logp: np.ndarray,end_logp: np.ndarray,weight: float) -> np.ndarray:
    """Maximize BIO + learned start/end evidence, without text or label rules.

    Start at character i predicts the B-tag's type, otherwise no-boundary.
    End at i predicts its entity type iff the next state leaves that entity.
    The final document character closes an entity. Weight zero is plain BIO.
    """
    if bio_logp.ndim!=2 or bio_logp.shape[1]!=25 or not np.isfinite(bio_logp).all() or not 0<=weight<=1:
        raise ValueError('Invalid BIO probabilities or boundary decoding weight')
    if weight==0:return constrained_viterbi(bio_logp)
    length=len(bio_logp)
    if start_logp.shape!=(length,13) or end_logp.shape!=(length,13) or not np.isfinite(start_logp).all() or not np.isfinite(end_logp).all():
        raise ValueError('Invalid learned boundary log probabilities')
    if not length:return np.zeros(0,dtype=np.uint8)
    state_labels=np.array([0,*[(tag-1)//2+1 for tag in range(1,25)]])
    begin_ids,inside_ids=np.arange(1,25,2),np.arange(2,25,2)
    starts=np.where(np.arange(25)%2==1,state_labels,0)
    emissions=bio_logp+weight*start_logp[:,starts]
    score=emissions[0].copy();score[2::2]=-np.inf
    back=np.zeros((length,25),dtype=np.uint8);new_states=np.array([0,*begin_ids])
    for index in range(1,length):
        ending=score+weight*end_logp[index-1,state_labels]
        best=int(ending.argmax())
        previous_inside=np.where(score[begin_ids]>=score[inside_ids],begin_ids,inside_ids)
        new=np.empty_like(score)
        new[new_states]=ending[best]+emissions[index,new_states]
        new[inside_ids]=score[previous_inside]+weight*end_logp[index-1,0]+emissions[index,inside_ids]
        back[index,new_states]=best;back[index,inside_ids]=previous_inside;score=new
    tags=np.zeros(length,dtype=np.uint8)
    tags[-1]=(score+weight*end_logp[-1,state_labels]).argmax()
    for index in range(length-1,0,-1):tags[index-1]=back[index,tags[index]]
    return tags


def scored_spans(log_probabilities: np.ndarray, labels: list[str],start_logp: np.ndarray | None=None,
                 end_logp: np.ndarray | None=None,boundary_weight: float=0.0) -> list[dict]:
    maximum = log_probabilities.max(1, keepdims=True)
    normalizer = maximum + np.log(np.exp(log_probabilities - maximum).sum(1, keepdims=True))
    normalized = log_probabilities - normalizer
    tags = (boundary_viterbi(normalized,start_logp,end_logp,boundary_weight)
            if start_logp is not None and boundary_weight else constrained_viterbi(normalized))
    spans = decode_tags(tags, labels)
    return [{'span': span, 'confidence': float(np.exp(normalized[np.arange(span[0],span[1]),
                         tags[span[0]:span[1]]]).mean())} for span in spans]

def filter_spans(scored: dict, threshold: float) -> dict:
    return {key: [row['span'] for row in value if row['confidence'] >= threshold] for key,value in scored.items()}
''')


[Markdown Cell]

### model.py

The foundation is asserted within 50M–300M unique parameters. Report total, active NER and trainable adaptation counts separately; PEFT intentionally trains fewer weights. The learned head combines contextual token features, character embeddings/flags and two five-character convolutions.

[Markdown Cell]

Exact character tagging for all four original architectures — source part 1

In [35]:
# [Code Cell]
append_module('model.py', r'''# CELL: Exact character tagging for all four original architectures
from dataclasses import replace
from pathlib import Path
import torch
from torch import Tensor,nn
from torch.nn import functional as F
from torch.nn.utils.rnn import pack_padded_sequence,pad_packed_sequence
from .parents import parent_class
from .adaptation import apply_adaptation

class CharacterNER(nn.Module):
    """Own Phase 1 features + learned character BIO classifier.

    token IDs/mask[B,T], character IDs/flags/token positions/mask[B,C]
    -> logits[B,C,25]. B uses its bidirectional encode(); A/C/D keep their
    original causal/hybrid/sparse forward features and window reset semantics.
    """
    def __init__(self,backbone: nn.Module,width: int,cfg):
        super().__init__(); self.backbone=backbone; self.family=cfg.family
        self.token_projection=nn.Linear(width,cfg.head_width)
        self.character_embedding=nn.Embedding(cfg.character_buckets,cfg.character_width)
        self.flag_embedding=nn.Embedding(64,8)
        self.fusion=nn.Linear(cfg.head_width+cfg.character_width+8,cfg.head_width)
        self.convolutions=nn.ModuleList(nn.Conv1d(cfg.head_width,cfg.head_width,5,padding=2) for _ in range(2))
        self.norms=nn.ModuleList(nn.LayerNorm(cfg.head_width) for _ in range(2))
        self.dropout=nn.Dropout(cfg.head_dropout); self.classifier=nn.Linear(cfg.head_width,25)
        # Initialize additions after the common head, so its tensors match controls.
        self.context_gru=None; self.context_projection=None
        if cfg.head_context=='token_bigru':
            self.context_gru=nn.GRU(cfg.head_width,cfg.head_width//2,batch_first=True,bidirectional=True)
            self.context_projection=nn.Linear(cfg.head_width,cfg.head_width)
            nn.init.zeros_(self.context_projection.weight); nn.init.zeros_(self.context_projection.bias)
        self.boundary_start=self.boundary_end=None
        if cfg.boundary_loss_weight:
            self.boundary_start=nn.Linear(cfg.head_width,13)
            self.boundary_end=nn.Linear(cfg.head_width,13)
        # Optional learned right/left CHARACTER context; no entity-specific rules.
        self.character_gru=self.character_projection=None
        if cfg.character_context:
            self.character_gru=nn.GRU(cfg.head_width,cfg.head_width//2,batch_first=True,bidirectional=True)
            self.character_projection=nn.Linear(cfg.head_width,cfg.head_width)
            nn.init.zeros_(self.character_projection.weight)
            nn.init.zeros_(self.character_projection.bias)

    def forward(self,input_ids: Tensor,attention_mask: Tensor,character_ids: Tensor,flags: Tensor,
                token_positions: Tensor,character_mask: Tensor,return_auxiliary: bool=False) -> Tensor | tuple[Tensor,Tensor,Tensor]:
        """Return BIO[B,C,25], or BIO/start/end logits when training auxiliaries.

''')


[Markdown Cell]

Exact character tagging for all four original architectures — source part 2

In [36]:
# [Code Cell]
append_module('model.py', r'''        The optional task GRU sees both directions inside a valid token window.
        Original backbone masks remain unchanged; packed lengths exclude padding.
        Boundaries are learned from train annotations, never from text rules.
        """
        hidden=self.backbone.encode(input_ids,attention_mask) if self.family=='b' else self.backbone.hidden_states(input_ids,attention_mask)
        projected=self.token_projection(hidden)
        if self.context_gru is not None:
            lengths=attention_mask.sum(1).to(device='cpu',dtype=torch.long)
            packed=pack_padded_sequence(projected,lengths,batch_first=True,enforce_sorted=False)
            packed_context,_=self.context_gru(packed)  # Independent zero state per window.
            context,_=pad_packed_sequence(packed_context,batch_first=True,total_length=projected.shape[1])
            projected=projected+self.context_projection(context)
        contextual=projected.gather(1,token_positions[...,None].expand(-1,-1,projected.shape[-1]))
        features=torch.cat((contextual,self.character_embedding(character_ids),self.flag_embedding(flags)),dim=-1)
        mask=character_mask[...,None]; features=F.gelu(self.fusion(features))*mask
        for convolution,norm in zip(self.convolutions,self.norms):
            update=convolution(features.transpose(1,2)).transpose(1,2)
            features=norm(features+self.dropout(F.gelu(update)))*mask
        if self.character_gru is not None:
            lengths=character_mask.sum(1).clamp_min(1).to(device='cpu',dtype=torch.long)
            packed=pack_padded_sequence(features,lengths,batch_first=True,enforce_sorted=False)
            context,_=self.character_gru(packed)  # State resets for every window.
            context,_=pad_packed_sequence(context,batch_first=True,total_length=features.shape[1])
            features=(features+self.character_projection(context))*mask
        dropped=self.dropout(features); logits=self.classifier(dropped)
        if return_auxiliary and self.boundary_start is not None:
            return logits,self.boundary_start(dropped),self.boundary_end(dropped)
        return logits

def build_model(job: dict,cfg,device: torch.device,load_parent: bool=True) -> CharacterNER:
    ParentConfig,ParentModel=parent_class(job['parent_source_dir'],cfg.family,job['parent_contract'])
    parent_cfg=replace(ParentConfig(**job['parent']['config']),activation_checkpointing=cfg.activation_checkpointing)
    # Meta initialization skips expensive random GRU QR initialization: every base
    # tensor is immediately replaced by real trained or resumed checkpoint weights.
    if load_parent:
        path=Path(job['parent']['checkpoint']); before=path.stat()
        if before.st_size!=job['parent']['checkpoint_bytes'] or before.st_mtime_ns!=job['parent']['checkpoint_mtime_ns']:
            raise ValueError('Parent checkpoint changed after verification')
        payload=torch.load(path,map_location='cpu',weights_only=True,mmap=True)
        with torch.device('meta'):
            backbone=ParentModel(parent_cfg)
        backbone.load_state_dict(payload['model'],strict=True,assign=True)
        del payload
    else:
        with torch.device('meta'):
            backbone=ParentModel(parent_cfg)
        # Preserve real nonpersistent RoPE caches while allocating empty meta weights.
        backbone._apply(lambda tensor: torch.empty_like(tensor,device=device) if tensor.is_meta else tensor.to(device))
''')


[Markdown Cell]

Exact character tagging for all four original architectures — source part 3

In [37]:
# [Code Cell]
append_module('model.py', r'''    foundation=sum(p.numel() for p in backbone.parameters())
    assert 50_000_000<foundation<300_000_000, f'Foundation boundary: {foundation}'
    capacity=parent_cfg.max_source_tokens if cfg.family=='b' else parent_cfg.max_seq_tokens
    if cfg.window_tokens>capacity:
        raise ValueError('NER token window exceeds the saved foundation context capacity')
    adapted=apply_adaptation(backbone,cfg,parent_cfg.d_model)
    # Paired trials use the same common head even when adapter rank consumes a
    # different number of random values. A new v2 local control is measured first.
    with torch.random.fork_rng(devices=[]):
        torch.random.default_generator.manual_seed(cfg.seed+1000)
        model=CharacterNER(backbone,parent_cfg.d_model,cfg).to(device)
    model.parameter_report=parameter_report(model,foundation,adapted)
    assert 50_000_000<model.parameter_report['total_unique_parameters']<300_000_000
    assert model.parameter_report['trainable_parameters']>0
    return model

def parameter_report(model,foundation: int,adapted: list[str]) -> dict:
    active=[(name,p) for name,p in model.named_parameters() if not (model.family=='b' and
            name.startswith(('backbone.decoder.','backbone.decoder_norm.','backbone.target_positions.')))]
    return {'foundation_unique_parameters':foundation,'total_unique_parameters':sum(p.numel() for p in model.parameters()),
            'active_ner_parameters':sum(p.numel() for _,p in active),
            'trainable_parameters':sum(p.numel() for p in model.parameters() if p.requires_grad),
            'adapted_modules':adapted,'inactive_decoder':model.family=='b'}

def forward_batch(model,batch: dict,device: torch.device,auxiliary: bool=False):
    keys=('input_ids','attention_mask','character_ids','flags','token_positions','character_mask')
    return model(**{key:batch[key].to(device,non_blocking=True) for key in keys},return_auxiliary=auxiliary)

def weighted_loss(logits: Tensor,targets: Tensor,weights: Tensor) -> Tensor:
    return F.cross_entropy(logits.float().transpose(1,2),targets,weight=weights,ignore_index=-100,reduction='sum')

''')


[Markdown Cell]

Globally normalized supervised BIO plus learned start/end objectives — source part 1

In [38]:
# [Code Cell]
append_module('model.py', r'''# CELL: Globally normalized supervised BIO plus learned start/end objectives
def boundary_objective(outputs,batch,weights,boundary_weights,normalizers,cfg,device):
    """Return weighted sums scaled to the BIO denominator for correct DDP means.

    L = BIO_sum/N_BIO + lambda/2*(start_sum/N_start + end_sum/N_end).
    Targets outside a window's owned characters, including dummy rows, are -100.
    Auxiliary heads predict 0/no-boundary or one of twelve learned entity types.
    """
    tags=outputs[0] if isinstance(outputs,tuple) else outputs
    targets=batch['targets'].to(device,non_blocking=True)
    loss=weighted_loss(tags,targets,weights)
    if cfg.boundary_loss_weight:
        if not isinstance(outputs,tuple) or len(outputs)!=3:
            raise ValueError('Missing trainable boundary logits')
        sums=[weighted_loss(logits,batch[key].to(device,non_blocking=True),boundary_weights)
              for logits,key in zip(outputs[1:],('start_targets','end_targets'))]
        loss=loss+cfg.boundary_loss_weight*.5*normalizers[0]*(sums[0]/normalizers[1]+sums[1]/normalizers[2])
    return loss
''')


[Markdown Cell]

### parents.py

Find a complete own Phase 1 output by original source and preprocessing identity. There is no fixed epoch or step requirement. Validate the latest real checkpoint cursor and source/config/Torch identity before selecting best/latest model weights.

[Markdown Cell]

Match a parent by source and data rather than a fixed epoch count — source part 1

In [39]:
# [Code Cell]
append_module('parents.py', r'''# CELL: Match a parent by source and data rather than a fixed epoch count
import hashlib
import importlib
import importlib.util
import json
import math
from pathlib import Path
import sys
import torch
from tokenizers import Tokenizer
from .io_utils import atomic_json, read_json, sha256_file

def source_hashes(directory: Path) -> dict[str,str]:
    return {path.name: sha256_file(path) for path in sorted(directory.glob('*.py'))}

def learning_choices(values: dict, contract: dict) -> dict:
    return {key:value for key,value in values.items() if key not in contract['fingerprint_exclusions']}

def reproduce_parent(values: dict, contract: dict, version: str) -> str:
    payload={'version':contract['runtime_version'],'config':learning_choices(values,contract),
             'data':contract['data_fingerprint'],'sources':contract['source_sha256'],'torch':version}
    return hashlib.sha256(json.dumps(payload,sort_keys=True).encode()).hexdigest()

def intact_pairs(root: Path) -> tuple[list, list]:
    """Check receipts/checkpoint bytes incrementally before mmap loading."""
    pairs=[]; warnings=[]
    for name in ('slot_0.pt','slot_1.pt','best.pt'):
        path=root/'checkpoints'/name; receipt_path=path.with_name(name+'.json')
        if not path.exists() and not receipt_path.exists():
            continue
        try:
            receipt=read_json(receipt_path)
            if any(type(receipt.get(key)) is not int or receipt[key]<0 for key in ('bytes','epoch','step','generation')):
                raise ValueError('Invalid receipt counters')
            if receipt['bytes']!=path.stat().st_size or receipt['sha256']!=sha256_file(path):
                raise ValueError('Size/SHA-256 mismatch')
            pairs.append((path,receipt))
        except (ValueError,OSError,KeyError,TypeError) as error:
            warnings.append(f'{path}: {error}')
    if not pairs:
        raise FileNotFoundError('No intact full checkpoint/receipt pair. ' + '; '.join(warnings))
    return pairs,warnings

def parent_candidates(cfg, contract: dict) -> list[dict]:
    """Return useful rejection reasons for every discovered run root."""
    roots=[Path(cfg.parent_run_dir)] if cfg.parent_run_dir else sorted({p.parent for p in Path(cfg.input_root).rglob('train_config.json')})
    rows=[]
    for root in roots:
''')


[Markdown Cell]

Match a parent by source and data rather than a fixed epoch count — source part 2

In [40]:
# [Code Cell]
append_module('parents.py', r'''        row={'run_dir':str(root),'reasons':[],'accepted':False}
        try:
            required=('train_config.json','data_report.json','training_summary.json',f'code/model_{cfg.family}/model.py')
            missing=[name for name in required if not (root/name).is_file()]
            if missing:
                row['reasons'].append('Missing: '+', '.join(missing))
            else:
                saved=read_json(root/'train_config.json'); summary=read_json(root/'training_summary.json')
                row.update(planned_epochs=saved.get('epochs'),completed_epochs=summary.get('epoch'),step=summary.get('step'),status=summary.get('status'))
                if type(saved.get('epochs')) is not int or saved['epochs']<1:
                    row['reasons'].append('Invalid configured epoch count')
                actual=source_hashes(root/f'code/model_{cfg.family}')
                changed=sorted(name for name in set(actual)|set(contract['source_sha256']) if actual.get(name)!=contract['source_sha256'].get(name))
                if changed:
                    row['reasons'].append('Original source differs: '+', '.join(changed))
                if read_json(root/'data_report.json').get('data_fingerprint')!=contract['data_fingerprint']:
                    row['reasons'].append('Wrong preprocessing identity for this model')
                names=[name for name in ('slot_0.pt','slot_1.pt','best.pt') if (root/'checkpoints'/name).is_file()
                       and (root/'checkpoints'/(name+'.json')).is_file()]
                row['checkpoint_pairs']=names
                if not names:
                    row['reasons'].append('Attach saved output with real .pt files and .pt.json receipts; .ipynb/prediction ZIP is insufficient')
                row['accepted']=not row['reasons']
        except (ValueError,OSError,KeyError,TypeError,AttributeError) as error:
            row['reasons'].append(f'{type(error).__name__}: {error}')
        rows.append(row)
    return rows

''')


[Markdown Cell]

Check real cursor/config/Torch identity before selecting weights — source part 1

In [41]:
# [Code Cell]
append_module('parents.py', r'''# CELL: Check real cursor/config/Torch identity before selecting weights
def validate_payload(payload: dict, receipt: dict, values: dict, contract: dict) -> None:
    required={'model','config','cursor','optimizer','scaler','scheduler','rng_by_rank','world_size'}
    if required-payload.keys():
        raise ValueError('Parent checkpoint lacks full recovery state')
    cursor=payload['cursor']
    if (any(type(cursor.get(key)) is not int or cursor[key]<0 for key in ('epoch','next_batch','global_step','optimizer_steps'))
            or payload['fingerprint']!=receipt['fingerprint'] or payload['step']!=receipt['step']
            or payload['epoch']!=receipt['epoch'] or payload['epoch']!=cursor['epoch']
            or payload['step']!=cursor['global_step'] or cursor['epoch']>values['epochs']
            or cursor['optimizer_steps']>cursor['global_step']
            or payload['scheduler']['steps']!=cursor['optimizer_steps']
            or payload['world_size']!=values['world_size'] or len(payload['rng_by_rank'])!=values['world_size']
            or learning_choices(payload['config'],contract)!=learning_choices(values,contract)):
        raise ValueError('Parent cursor/config/scheduler/RNG/receipt disagree')

def verify_parent(cfg, contract: dict) -> tuple[Path,dict]:
    diagnostics=parent_candidates(cfg,contract)
    atomic_json(Path(cfg.run_dir)/'parent_input_report.json',{'candidates':diagnostics})
    print(json.dumps({'parent_candidates':diagnostics},indent=2),flush=True)
    accepted=[Path(row['run_dir']) for row in diagnostics if row['accepted']]
    if len(accepted)!=1:
        raise ValueError(f'Found {len(accepted)} matching Model {cfg.family.upper()} outputs. Attach ONE full saved '
                         'output or set parent_run_dir to its exact printed run_dir; see parent_input_report.json.')
    root=accepted[0]; values=read_json(root/'train_config.json')
    pairs,warnings=intact_pairs(root)
    latest=max(pairs,key=lambda item:(item[1]['step'],item[1]['generation']))
    payload=torch.load(latest[0],map_location='cpu',weights_only=True,mmap=True)
    validate_payload(payload,latest[1],values,contract)
    versions=list(contract['recorded_torch_candidates'])+[str(torch.__version__)]
    for name in ('continuation_runtime_identity.json','training_runtime_identity.json'):
        path=root/name
        if path.is_file():
            record=read_json(path)
            if isinstance(record.get('torch'),str):
                versions.insert(0,record['torch'])
    matched=next((v for v in dict.fromkeys(versions) if reproduce_parent(values,contract,v)==payload['fingerprint']),None)
    if matched is None:
        raise ValueError('Cannot reproduce original config/source/data/Torch fingerprint. Attach the matching original runtime and tokenizer; do not disable this check.')
    cursor=payload['cursor']
    if cursor['epoch']!=values['epochs'] or cursor['next_batch']!=0:
        raise ValueError(f"Parent has {cursor['epoch']}/{values['epochs']} completed epochs, next_batch={cursor['next_batch']}. Finish/save its continuation before starting a new NER experiment.")
    report=read_json(root/'data_report.json')
    examples=report.get('train_examples')
    coverage=(isinstance(report.get('eligible_source_tokens'),dict) if cfg.family=='b' else report.get('full_window_coverage') is True)
    if type(examples) is not int or examples<1 or not coverage:
        raise ValueError('Parent data report lacks a positive exhaustive training-window count')
    global_batch=values['world_size']*values['micro_batch_size']*values['accumulation_steps']
''')


[Markdown Cell]

Check real cursor/config/Torch identity before selecting weights — source part 2

In [42]:
# [Code Cell]
append_module('parents.py', r'''    expected_steps=math.ceil(examples/global_batch)*cursor['epoch']
    if cursor['global_step']!=expected_steps:
        raise ValueError(f"Parent counters disagree with its exhaustive window plan: expected {expected_steps} boundaries, got {cursor['global_step']}")
    summary=read_json(root/'training_summary.json')
    if summary.get('status')!='complete' or summary.get('epoch')!=cursor['epoch'] or summary.get('step')!=cursor['global_step'] or summary.get('fingerprint')!=payload['fingerprint']:
        raise ValueError('Parent training summary disagrees with its latest intact checkpoint')
    chosen=next((item for item in pairs if item[0].name=='best.pt'),latest) if cfg.parent_checkpoint=='best' else latest
    selected=torch.load(chosen[0],map_location='cpu',weights_only=True,mmap=True)
    validate_payload(selected,chosen[1],values,contract)
    if selected['fingerprint']!=payload['fingerprint']:
        raise ValueError('Selected parent best/latest belongs to another run')
    for name,tensor in selected['model'].items():
        if tensor.is_floating_point():
            if tensor.dtype!=torch.float32 or not all(bool(torch.isfinite(c).all()) for c in tensor.reshape(-1).split(1_000_000)):
                raise ValueError(f'Nonfinite/non-FP32 master weights: {name}')
    result={'family':cfg.family,'config':values,'completed_epochs':cursor['epoch'],'planned_epochs':values['epochs'],
            'checkpoint':str(chosen[0]),'checkpoint_sha256':chosen[1]['sha256'],'checkpoint_bytes':chosen[1]['bytes'],
            'checkpoint_mtime_ns':chosen[0].stat().st_mtime_ns,'checkpoint_epoch':chosen[1]['epoch'],'checkpoint_step':chosen[1]['step'],
            'selection':cfg.parent_checkpoint,'original_fingerprint':payload['fingerprint'],'original_torch':matched,
            'data_fingerprint':contract['data_fingerprint'],'source_sha256':contract['source_sha256'],'checkpoint_warnings':warnings}
    print(f"Verified Model {cfg.family.upper()}: {cursor['epoch']} completed epochs; selected {chosen[0].name} at epoch {chosen[1]['epoch']}",flush=True)
    return root,result

''')


[Markdown Cell]

Find exactly matching small preprocessing artifacts without reading LM binaries — source part 1

In [43]:
# [Code Cell]
append_module('parents.py', r'''# CELL: Find exactly matching small preprocessing artifacts without reading LM binaries
def tokenizer_inputs(cfg, contract: dict) -> tuple[Path,Path,dict]:
    metadata=[Path(cfg.metadata_path)] if cfg.metadata_path else sorted(Path(cfg.input_root).rglob('metadata.json'))
    matches=[]
    required=('train.bin','val.bin','tokenizer.json','train_documents.jsonl','validation_documents.jsonl','validation_pairs.jsonl')
    for path in metadata:
        if not path.is_file():
            continue
        try:
            value=read_json(path)
            hashes={name:value['artifacts'][name]['sha256'] for name in required}
            hashes['metadata.json']=sha256_file(path)
            fingerprint=hashlib.sha256(json.dumps(hashes,sort_keys=True).encode()).hexdigest()
            if fingerprint==contract['data_fingerprint']:
                matches.append((path,value))
        except (KeyError,TypeError,ValueError,OSError):
            continue
    if not matches:
        raise FileNotFoundError(f'Missing matching Model {cfg.family.upper()} metadata.json. Use its original preprocessing output, not another token-budget export; large .bin files are unnecessary.')
    meta,value=matches[0]; expected=value['artifacts']['tokenizer.json']['sha256']
    candidates=[Path(cfg.tokenizer_path)] if cfg.tokenizer_path else [meta.parent/'tokenizer.json',*Path(cfg.input_root).rglob('tokenizer.json')]
    token=next((p for p in candidates if p.is_file() and sha256_file(p)==expected),None)
    if token is None:
        raise FileNotFoundError('No tokenizer.json matches the verified metadata; attach the original tokenizer')
    tokenizer=Tokenizer.from_file(str(token))
    specials={'<pad>':0,'<s>':1,'</s>':2,'<unk>':3,'<sep>':4,'<mask>':5}
    if tokenizer.get_vocab_size()!=32768 or any(tokenizer.token_to_id(k)!=v for k,v in specials.items()):
        raise ValueError('Unexpected parent tokenizer vocabulary/special IDs')
    return token,meta,{'tokenizer_sha256':expected,'metadata_sha256':sha256_file(meta)}

def parent_class(directory: str, family: str, contract: dict):
    """Import only the checksum-verified original config and model implementation."""
    root=Path(directory)
    if source_hashes(root)!=contract['source_sha256']:
        raise ValueError('Parent source changed after verification')
    package='verified_parent_'+family+'_'+contract['source_sha256']['model.py'][:12]
    if package not in sys.modules:
        spec=importlib.util.spec_from_file_location(package,root/'__init__.py',submodule_search_locations=[str(root)])
        module=importlib.util.module_from_spec(spec); sys.modules[package]=module; spec.loader.exec_module(module)
    config=importlib.import_module(package+'.config'); model=importlib.import_module(package+'.model')
    return config.TrainConfig,getattr(model,contract['class_name'])
''')


[Markdown Cell]

### predict.py

Fuse complete document windows, preserve blank IDs, check original-character bounds, compare validation with the provided official evaluator and verify the final root prediction.json ZIP.

[Markdown Cell]

Whole-document prediction with overlapping-context fusion — source part 1

In [44]:
# [Code Cell]
append_module('predict.py', r'''# CELL: Whole-document prediction with overlapping-context fusion
import argparse
import json
from pathlib import Path
import subprocess
import sys
import zipfile
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from .config import NERConfig
from .data import WindowDataset, collate_windows
from .io_utils import atomic_json, read_json, records, sha256_file
from .metrics import filter_spans, metrics, scored_spans
from .model import build_model, forward_batch

@torch.inference_mode()
def predict_partition(model, job: dict, cfg, split: str, device: torch.device, rank: int = 0, world_size: int = 1) -> dict:
    """Evaluate each full document on one rank; no DDP forward collectives."""
    model.eval()
    dataset = WindowDataset(job['cache']['directory'], split, 1)
    output = {}
    for doc, entry in enumerate(tqdm(dataset.documents, desc=f'{cfg.experiment_id} {split} documents', disable=rank != 0)):
        if doc % world_size != rank:
            continue
        if entry['blank']:
            output[entry['id']] = []; continue
        scores = np.zeros((entry['characters'], 25), dtype=np.float32)
        use_boundaries=bool(cfg.boundary_loss_weight and cfg.boundary_decoder_weight)
        boundary_scores=[np.zeros((entry['characters'],13),dtype=np.float32) for _ in range(2)] if use_boundaries else None
        total_weight = np.zeros(entry['characters'], dtype=np.float32)
        indices = np.flatnonzero(dataset.windows[:,0] == doc).tolist()
        if not indices:
            raise ValueError('Nonblank document has no prediction windows.')
        loader = DataLoader(torch.utils.data.Subset(dataset, indices), batch_size=cfg.micro_batch_size,
                            collate_fn=collate_windows, num_workers=0, pin_memory=device.type == 'cuda')
        for batch in loader:
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
                outputs = forward_batch(model, batch, device,auxiliary=use_boundaries)
                logits=outputs[0] if isinstance(outputs,tuple) else outputs
            logp = logits.float().log_softmax(-1).cpu().numpy()
            boundary_logp=[value.float().log_softmax(-1).cpu().numpy() for value in outputs[1:]] if use_boundaries else None
            if not np.isfinite(logp).all():
                raise ValueError('Nonfinite NER logits. No prediction ZIP will be published.')
            for index, meta in enumerate(batch['metadata']):
                length = meta['end'] - meta['start']
                positions = np.arange(length)
''')


[Markdown Cell]

Whole-document prediction with overlapping-context fusion — source part 2

In [45]:
# [Code Cell]
append_module('predict.py', r'''                weights = np.minimum(positions + 1, length - positions).clip(1,128).astype(np.float32)
                section = slice(meta['start'], meta['end'])
                scores[section] += logp[index,:length] * weights[:,None]
                if use_boundaries:
                    for values,probabilities in zip(boundary_scores,boundary_logp):
                        values[section]+=probabilities[index,:length]*weights[:,None]
                total_weight[section] += weights
        if (total_weight == 0).any():
            raise ValueError('Uncovered document characters. Full-document inference is required.')
        output[entry['id']] = scored_spans(scores / total_weight[:,None],job['labels'],
            start_logp=boundary_scores[0]/total_weight[:,None] if use_boundaries else None,
            end_logp=boundary_scores[1]/total_weight[:,None] if use_boundaries else None,
            boundary_weight=cfg.boundary_decoder_weight if use_boundaries else 0)
    dataset.close()
    return output

''')


[Markdown Cell]

Verify exact span outputs and compare with the official evaluator — source part 1

In [46]:
# [Code Cell]
append_module('predict.py', r'''# CELL: Verify exact span outputs and compare with the official evaluator
def validate_predictions(predictions: dict, prepared: Path, split: str, labels: list[str]) -> None:
    rows = {row['id']: row for row in records(prepared / (split + '.jsonl'))}
    if set(rows) != set(predictions):
        raise ValueError('Missing/extra document IDs.')
    for key, spans in predictions.items():
        seen = set()
        for span in spans:
            if not isinstance(span,list) or len(span) != 3:
                raise ValueError('Expected [start,end,label] triples.')
            start, end, label = span
            if type(start) is not int or type(end) is not int or not 0 <= start < end <= len(rows[key]['text']) or label not in labels:
                raise ValueError(f'Invalid original-character span: {key}/{span}')
            if tuple(span) in seen:
                raise ValueError('Duplicate entity triple.')
            seen.add(tuple(span))
        if not rows[key]['text'].strip() and spans:
            raise ValueError('Blank documents must have empty predictions.')

def official_validation(job: dict, prediction_path: Path, calculated: dict) -> str:
    prepared = Path(job['prepared_dir'])
    result = subprocess.run([sys.executable, str(prepared/'evaluate_phase2.py'), '--truth',
                str(prepared/'validation_gold.json'), '--prediction', str(prediction_path),
                '--labels', str(prepared/'labels.txt')], capture_output=True, text=True, check=True)
    line = next(line for line in result.stdout.splitlines() if line.startswith('Strict Macro-F1:'))
    if abs(float(line.split(':')[1]) - calculated['strict_macro_f1']) > 1e-6:
        raise ValueError('Internal metrics differ from the supplied official evaluator.')
    return result.stdout

''')


[Markdown Cell]

Load the best actual NER weights and create the E01 submission — source part 1

In [47]:
# [Code Cell]
append_module('predict.py', r'''# CELL: Load the best actual NER weights and create the E01 submission
def export_best(job: dict, cfg) -> dict:
    from .state import Checkpoints
    manager = Checkpoints(Path(cfg.run_dir), job['fingerprint'], cfg.min_free_bytes)
    candidate = manager.best()
    if candidate is None:
        raise RuntimeError('No completed NER validation checkpoint exists yet. Resume training from the saved E01 output first.')
    path, receipt = candidate
    payload = torch.load(path, map_location='cpu', weights_only=True, mmap=True)
    if payload['fingerprint'] != job['fingerprint']:
        raise ValueError('NER checkpoint identity mismatch.')
    device = torch.device('cuda',0)
    torch.set_num_threads(2)
    model = build_model(job, cfg, device, load_parent=False)
    model.load_state_dict(payload['model'], strict=True)
    threshold = payload['cursor']['best_threshold']
    policy=read_json(Path(cfg.run_dir)/'frozen_calibration.json') if (Path(cfg.run_dir)/'frozen_calibration.json').exists() else None
    output = Path(cfg.run_dir) / 'predictions'; output.mkdir(exist_ok=True)
    reports = {}
    partitions=(('train','test') if cfg.report_train_f1 else ('test',)) if cfg.selection_mode=='public_refit' else (('train','validation','test') if cfg.report_train_f1 else ('validation','test'))
    for split in partitions:
        scored = predict_partition(model, job, cfg, split, device)
        if policy:
            from .calibration import apply_policy
            predictions=apply_policy(scored,policy['policy'])
        else:
            predictions=filter_spans(scored,threshold)
        validate_predictions(predictions, Path(job['prepared_dir']), split, job['labels'])
        atomic_json(output / (split + '_prediction.json'), predictions)
        reports[split] = {'examples': len(predictions), 'entities': sum(map(len,predictions.values()))}
        if split=='train':
            from .io_utils import records
            gold={row['id']:row['entities'] for row in records(Path(job['prepared_dir'])/'train.jsonl')}
            train_metrics=metrics(gold,predictions,job['labels'])
            train_metrics['scope']='In-sample training fit; not held-out performance'
            atomic_json(output/'train_metrics.json',train_metrics)
        if split == 'validation':
            calculated = metrics(read_json(Path(job['prepared_dir'])/'validation_gold.json'), predictions, job['labels'])
            atomic_json(output/'validation_metrics.json', calculated)
            atomic_json(output/'validation_scored_spans.json',scored)
            atomic_json(output/'validation_unfiltered_metrics.json',metrics(
                read_json(Path(job['prepared_dir'])/'validation_gold.json'),filter_spans(scored,0),job['labels']))
            log = official_validation(job, output/(split+'_prediction.json'), calculated)
            (output/'official_validation.txt').write_text(log,encoding='utf-8')
    prediction_path = output/'prediction.json'
    atomic_json(prediction_path, read_json(output/'test_prediction.json'))
    archive_path = Path(cfg.run_dir)/('submission_'+cfg.experiment_id+'.zip')
    temporary = archive_path.with_name(archive_path.name+'.partial')
''')


[Markdown Cell]

Load the best actual NER weights and create the E01 submission — source part 2

In [48]:
# [Code Cell]
append_module('predict.py', r'''    with zipfile.ZipFile(temporary,'w',zipfile.ZIP_DEFLATED) as archive:
        archive.write(prediction_path,arcname='prediction.json')
    with zipfile.ZipFile(temporary) as archive:
        if archive.namelist() != ['prediction.json'] or archive.testzip() is not None:
            raise ValueError('Submission ZIP verification failed.')
        packaged = json.loads(archive.read('prediction.json'))
        validate_predictions(packaged, Path(job['prepared_dir']), 'test', job['labels'])
    import os
    os.replace(temporary,archive_path)
    result = {'status':'submission_verified', 'experiment_id':cfg.experiment_id, 'method':cfg.method,
              'family':cfg.family,'parent_completed_epochs':job['parent']['completed_epochs'],
              'parameter_report':model.parameter_report,
              'checkpoint_sha256':receipt['sha256'], 'checkpoint_epoch':payload['epoch'],
              'parent_checkpoint_sha256':job['parent']['checkpoint_sha256'],
              'tokenizer_sha256':job['parent']['tokenizer_sha256'], 'split_sha256':cfg.expected_split_sha256,
              'confidence_threshold':threshold, 'validation':reports.get('validation',{'examples':0,'scope':'No held-out split after public refit'}), 'test':reports['test'],
              'selection_mode':cfg.selection_mode,'checkpoint_policy':'fixed final epoch' if cfg.selection_mode=='public_refit' else 'held-out best',
              'validation_macro_f1':None if cfg.selection_mode=='public_refit' else read_json(output/'validation_metrics.json')['strict_macro_f1'],
              'teacher':{key:value for key,value in job.get('teacher',{}).items() if key not in ('directory','relative_directory')},
              'zip_path':str(archive_path), 'zip_sha256':sha256_file(archive_path),
              'prediction_sha256':sha256_file(prediction_path), 'private_test_score':None}
    atomic_json(Path(cfg.run_dir)/'submission_receipt.json',result)
    return result

def main() -> None:
    parser = argparse.ArgumentParser(); parser.add_argument('--job',required=True)
    args = parser.parse_args(); job=read_json(args.job)
    cfg=NERConfig(**{**job['config'],'confidence_thresholds':tuple(job['config']['confidence_thresholds'])})
    cfg.validate()
    from .io_utils import run_lock
    with run_lock(Path(cfg.run_dir)):
        print(json.dumps(export_best(job,cfg),indent=2),flush=True)

if __name__ == '__main__':
    main()
''')


[Markdown Cell]

### prepare.py

Verify inputs, detect task recovery, commit one target epoch, build exhaustive caches and exercise the actual full-size task architecture before a long run.

[Markdown Cell]

Portable recovery discovery and an idempotent extendable epoch plan — source part 1

In [49]:
# [Code Cell]
append_module('prepare.py', r'''# CELL: Portable recovery discovery and an idempotent extendable epoch plan
from dataclasses import asdict,replace
import argparse
import json
from pathlib import Path
import shutil
import torch
from .artifacts import prepared_input
from .parents import parent_class,source_hashes,tokenizer_inputs,verify_parent
from .config import NERConfig
from .data import build_cache,collate_windows,WindowDataset
from .io_utils import atomic_json,identity,read_json,run_lock,sha256_file
from .model import build_model,forward_batch,weighted_loss,boundary_objective
from .state import Checkpoints

VERSION='cs781-ner-repair-v2'

def require_hardware() -> None:
    if not torch.cuda.is_available() or torch.cuda.device_count()!=2 or not torch.distributed.is_nccl_available():
        raise RuntimeError('Select GPU T4 x2 in Kaggle; CUDA/NCCL and exactly two GPUs are required')
    for rank in range(2):
        name=torch.cuda.get_device_name(rank)
        if 'T4' not in name:
            raise RuntimeError(f'Expected a T4; GPU {rank}: {name}')
        print(f'GPU {rank}: {name}',flush=True)

def resume_jobs(cfg) -> list[tuple[Path,dict]]:
    local=Path(cfg.run_dir)/'job.json'
    if local.is_file():
        paths=[local]
    elif cfg.resume_run_dir:
        paths=[Path(cfg.resume_run_dir)/'job.json']
    else:
        paths=sorted(Path(cfg.input_root).rglob('job.json'))
    jobs=[]
    for path in paths:
        value=read_json(path)
        if value.get('config',{}).get('experiment_id')!=cfg.experiment_id:
            if cfg.resume_run_dir:
                raise ValueError('Explicit resume output belongs to another experiment_id')
            continue
        if value.get('version')!=VERSION:
            raise ValueError('Resume output uses an older runtime. Keep its original notebook; start this flexible version in a new experiment_id/run_dir.')
        saved=NERConfig(**{**value['config'],'confidence_thresholds':tuple(value['config']['confidence_thresholds'])})
        if saved.choices()!=cfg.choices():
            raise ValueError('Resume learning/architecture settings changed. Keep them unchanged; use additional_epochs/target_total_epochs for more training.')
        if source_hashes(path.parent/'code/ner_repair')!=value['runtime_source_sha256']:
            raise ValueError('Saved Phase 2 runtime files are incomplete/changed')
''')


[Markdown Cell]

Portable recovery discovery and an idempotent extendable epoch plan — source part 2

In [50]:
# [Code Cell]
append_module('prepare.py', r'''        jobs.append((path.parent,value))
    if len({job['fingerprint'] for _,job in jobs})>1:
        raise ValueError('Different NER experiments share this ID. Set resume_run_dir to one exact saved output.')
    return jobs

def training_plan(cfg, cursor: dict | None, old_plan: dict | None, local: bool) -> dict:
    """An added-epochs request is committed once per working output, not each rerun."""
    completed=cursor['epoch'] if cursor else 0
    previous_target=old_plan['target_epoch'] if old_plan else cfg.max_epochs
    extending=False
    if cfg.target_total_epochs is not None:
        target=cfg.target_total_epochs
        extending=target>previous_target or bool(cursor and cursor['complete'] and target>completed) or bool(old_plan and old_plan.get('extension_requested'))
    elif local and old_plan:
        target=previous_target; extending=old_plan.get('extension_requested',False)
    elif cursor and cfg.additional_epochs:
        if not cursor['complete']:
            target=previous_target  # Finish the previously saved plan before adding again.
            extending=old_plan.get('extension_requested',False)
        else:
            target=completed+cfg.additional_epochs; extending=True
    elif cfg.additional_epochs:
        raise ValueError('additional_epochs requires an existing completed Phase 2 NER output; use max_epochs for a first run')
    else:
        target=previous_target
    if target<completed or (target==completed and cursor and cursor['next_batch']):
        raise ValueError('Requested target is behind the saved epoch/batch cursor')
    return {'target_epoch':target,'start_completed_epochs':completed,'extension_requested':extending,
            'schedule_policy':'restore the saved schedule; hold its final LR after the original horizon',
            'early_stopping_active':cfg.stop_early and not extending}

''')


[Markdown Cell]

Assemble verified parent/tokenizer, complete data, and optional task recovery — source part 1

In [51]:
# [Code Cell]
append_module('prepare.py', r'''# CELL: Assemble verified parent/tokenizer, complete data, and optional task recovery
def prepare_job(cfg,catalog: dict) -> dict:
    cfg.validate(); run=Path(cfg.run_dir); run.mkdir(parents=True,exist_ok=True)
    prepared=prepared_input(cfg); contract=catalog[cfg.family]
    jobs=resume_jobs(cfg)
    if len(jobs)==1 and jobs[0][0].resolve()==run.resolve():
        # A previous successful/failed shape check may have prepared a job before
        # the training cell was ever run. Only a wholly checkpoint-free local
        # preparation can be rebuilt; saved or corrupt recovery is never ignored.
        recovery_files=list((run/'checkpoints').glob('*'))
        if not recovery_files and not (run/'training_summary.json').exists():
            jobs=[]
    previous=None; resume=None; cursor=None
    if jobs:
        previous=max(jobs,key=lambda item:read_json(item[0]/'training_summary.json').get('step',-1) if (item[0]/'training_summary.json').is_file() else -1)
        previous_root,prior=previous
        manager=Checkpoints(run,prior['fingerprint'],cfg.min_free_bytes,[str(root) for root,_ in jobs])
        resume=manager.latest()
        if resume is None:
            raise ValueError('Attached NER job lacks intact checkpoints. Refusing to restart from scratch.')
        payload=manager.load(resume); cursor=dict(payload['cursor']); del payload
        parent=dict(prior['parent'])
        source_root=previous_root/'parent_source'/f'model_{cfg.family}'
        token=previous_root/'tokenizer/tokenizer.json'; meta=previous_root/'tokenizer/metadata.json'
        if source_hashes(source_root)!=contract['source_sha256'] or sha256_file(token)!=parent['tokenizer_sha256'] or sha256_file(meta)!=parent['metadata_sha256']:
            raise ValueError('NER output lacks its immutable original parent source/tokenizer snapshot')
    else:
        parent_root,parent=verify_parent(cfg,contract)
        token,meta,token_identity=tokenizer_inputs(cfg,contract); parent.update(token_identity)
        source_root=parent_root/f'code/model_{cfg.family}'
    sources=run/'parent_source'/f'model_{cfg.family}'; sources.mkdir(parents=True,exist_ok=True)
    for name,digest in contract['source_sha256'].items():
        destination=sources/name
        if destination.exists() and sha256_file(destination)!=digest:
            raise ValueError('Existing parent snapshot differs; use a new experiment directory')
        if (source_root/name).resolve()!=destination.resolve():
            shutil.copyfile(source_root/name,destination)
    small=run/'tokenizer'; small.mkdir(exist_ok=True)
    for source,name in ((token,'tokenizer.json'),(meta,'metadata.json')):
        if source.resolve()!=(small/name).resolve():
            shutil.copyfile(source,small/name)
    cache=build_cache(prepared,small/'tokenizer.json',cfg,parent)
    plan=training_plan(cfg,cursor,previous[1].get('training_plan') if previous else None,
                       bool(previous and previous[0].resolve()==run.resolve()))
    cfg=replace(cfg,max_epochs=plan['target_epoch'])
    runtime=source_hashes(Path(__file__).parent)
    if previous and runtime!=previous[1]['runtime_source_sha256']:
        raise ValueError('Exact resume requires the same flexible notebook runtime version')
''')


[Markdown Cell]

Assemble verified parent/tokenizer, complete data, and optional task recovery — source part 2

In [52]:
# [Code Cell]
append_module('prepare.py', r'''    job={'version':VERSION,'config':asdict(cfg),'prepared_dir':str(prepared),'parent':parent,'parent_contract':contract,
         'environment':{'torch':str(torch.__version__),**cache['dependencies']},
         'parent_source_dir':str(sources),'tokenizer_path':str(small/'tokenizer.json'),'cache':cache,'labels':cache['labels'],
         'runtime_source_sha256':runtime,'resume_roots':[str(root) for root,_ in jobs],
         'training_plan':plan,'resume_checkpoint':str(resume[0]) if resume else None}
    if cfg.method=='distillation':
        from .distill import prepare_teacher_cache
        prior_teacher=dict(previous[1]['teacher']) if previous and previous[1].get('teacher') else None
        if prior_teacher:
            prior_teacher['directory']=str(previous[0]/prior_teacher['relative_directory'])
        job['teacher']=prepare_teacher_cache(job,cfg,catalog,prior_teacher)
    fingerprint=identity({'version':VERSION,'config':cfg.choices(),'runtime':runtime,'torch':str(torch.__version__),
                          'parent':parent['checkpoint_sha256'],'sources':parent['source_sha256'],
                          'tokenizer':parent['tokenizer_sha256'],'split':cfg.expected_split_sha256,
                          'cache':cache['identity'],'teacher':job.get('teacher',{}).get('identity')})
    if previous and fingerprint!=previous[1]['fingerprint']:
        raise ValueError('NER source/Torch/tokenizer/cache/teacher fingerprint changed. Use the original environment to resume.')
    job['fingerprint']=fingerprint
    atomic_json(run/'task_config.json',asdict(cfg)); atomic_json(run/'parent_manifest.json',parent)
    atomic_json(run/'environment.json',job['environment'])
    atomic_json(run/'job.json',job); atomic_json(run/'training_plan.json',plan)
    atomic_json(run/'data_alignment_report.json',{key:cache[key] for key in ('window_counts','all_assigned_documents_used',
                'every_training_character_owned_once','gold_character_roundtrip','text_normalization','head')})
    print(json.dumps({'parent_completed_epochs':parent['completed_epochs'],'NER_plan':plan},indent=2),flush=True)
    return job

''')


[Markdown Cell]

Verify the full-size model and task gradients before launching workers — source part 1

In [53]:
# [Code Cell]
append_module('prepare.py', r'''# CELL: Verify the full-size model and task gradients before launching workers
def sanity_check(job,cfg) -> dict:
    device=torch.device('cuda',0); torch.set_num_threads(2)
    model=build_model(job,cfg,device,load_parent=not bool(job['resume_checkpoint']))
    if job['resume_checkpoint']:
        state=torch.load(job['resume_checkpoint'],map_location='cpu',weights_only=True,mmap=True)
        model.load_state_dict(state['model'],strict=True); del state
    model.train()  # cuDNN GRU backward requires training mode, including frozen C GRUs.
    dataset=WindowDataset(job['cache']['directory'],'train',1)
    batch=collate_windows([dataset[0]])
    outputs=forward_batch(model,batch,device,auxiliary=bool(cfg.boundary_loss_weight))
    logits=outputs[0] if isinstance(outputs,tuple) else outputs
    assert logits.shape==(*batch['character_ids'].shape,25) and bool(torch.isfinite(logits).all())
    weights=torch.tensor(job['cache']['class_weights'],device=device)
    boundary_weights=torch.tensor(job['cache']['boundary_class_weights'],device=device)
    targets=batch['targets']; mask=targets!=-100
    normalizers=(float(weights[targets[mask].to(device)].sum()),
        float(boundary_weights[batch['start_targets'][mask].to(device)].sum()),
        float(boundary_weights[batch['end_targets'][mask].to(device)].sum()))
    loss=boundary_objective(outputs,batch,weights,boundary_weights,normalizers,cfg,device); loss.backward()
    if cfg.boundary_loss_weight:
        assert model.boundary_start.weight.grad is not None and model.boundary_end.weight.grad is not None
        assert torch.isfinite(model.boundary_start.weight.grad).all() and torch.isfinite(model.boundary_end.weight.grad).all()
    if cfg.head_context=='token_bigru':
        assert model.context_projection.weight.grad is not None and torch.isfinite(model.context_projection.weight.grad).all()
    assert bool(torch.isfinite(loss)) and model.classifier.weight.grad is not None
    trainable=[p for p in model.backbone.parameters() if p.requires_grad]
    assert trainable and any(p.grad is not None and bool(torch.isfinite(p.grad).all()) for p in trainable)
    report={**model.parameter_report,'logit_shape':list(logits.shape),'finite_logits_and_loss':True,
            'backbone_adaptation_and_head_gradients':'passed','scope':'Shape/gradient check, not an NER accuracy measurement'}
    atomic_json(Path(cfg.run_dir)/'sanity_report.json',report)
    dataset.close(); del model,logits,loss; torch.cuda.empty_cache()
    return report

def main() -> None:
    parser=argparse.ArgumentParser(); parser.add_argument('--config',required=True); parser.add_argument('--catalog',required=True)
    args=parser.parse_args(); cfg=NERConfig.load(args.config); require_hardware()
    with run_lock(Path(cfg.run_dir)):
        job=prepare_job(cfg,read_json(args.catalog))
        effective=NERConfig.load(Path(cfg.run_dir)/'task_config.json')
        print(json.dumps(sanity_check(job,effective),indent=2),flush=True)

if __name__=='__main__':
    main()
''')


[Markdown Cell]

### public_data.py

Freeze settings on 60/10, then refit all 70 public training documents; private test excluded.

[Markdown Cell]

Explicit final refit on every released training document — source part 1

In [54]:
# [Code Cell]
append_module('public_data.py', r'''# CELL: Explicit final refit on every released training document
"""Derive a separate 70/0/35 bundle; preserve original 60/10 artifacts unchanged."""
from pathlib import Path
import shutil
from .io_utils import atomic_json,identity,read_json,records,sha256_file


def public_refit_bundle(original: Path,output: Path) -> tuple[Path,dict]:
    """Combine the two public training partitions, excluding every test ID/group."""
    manifest=read_json(original/'prepared_manifest.json')
    train=list(records(original/'train.jsonl'));validation=list(records(original/'validation.jsonl'))
    test=list(records(original/'test.jsonl'));combined=train+validation
    if len(train)!=60 or len(validation)!=10 or len(test)!=35:raise ValueError('Unexpected prepared document counts')
    if len({r['id'] for r in combined})!=70:raise ValueError('Duplicate released training IDs')
    if {r['id'] for r in combined}&{r['id'] for r in test}:raise ValueError('Test ID enters public refit')
    if {r['group_sha256'] for r in combined}&{r['group_sha256'] for r in test}:raise ValueError('Test document group overlaps public training')
    if any('entities' in r for r in test):raise ValueError('Private test annotations must never be present')
    output.mkdir(parents=True,exist_ok=True)
    expected={'train':[r['id'] for r in combined],'validation':[],'test':[r['id'] for r in test]}
    split_hash=identity(expected)
    if (output/'prepared_manifest.json').exists():
        existing=read_json(output/'prepared_manifest.json')
        if existing.get('parent_prepared_identity')!=manifest['identity'] or existing['split_sha256']!=split_hash:
            raise ValueError('Existing full-public preparation changed')
        for name,receipt in existing['artifacts'].items():
            if sha256_file(output/name)!=receipt['sha256']:raise ValueError('Full-public artifact changed')
        return output,existing
    # Preserve each original JSONL row byte-for-byte. Nothing is normalized.
    (output/'train.jsonl').write_bytes((original/'train.jsonl').read_bytes()+(original/'validation.jsonl').read_bytes())
    (output/'validation.jsonl').write_bytes(b'')
    for name in ('test.jsonl','fold3-text.json','labels.txt','evaluate_phase2.py'):
        shutil.copyfile(original/name,output/name)
    atomic_json(output/'validation_gold.json',{});atomic_json(output/'validation_text.json',{})
    atomic_json(output/'split_manifest.json',{'ids':expected,'parent_split_sha256':manifest['split_sha256'],
        'split_unit':'Final all-public refit; no held-out validation remains'})
    atomic_json(output/'input_manifest.json',{'parent_identity':manifest['identity'],'parent_artifacts':manifest['artifacts']})
    atomic_json(output/'data_audit.json',{'public_training_documents':70,'validation_documents':0,'test_documents':35,
        'training_characters':sum(len(r['text']) for r in combined),'training_entities':sum(len(r['entities']) for r in combined),
        'test_gold_used':False,'normalization':'none','local_training_metrics_are_in_sample':True})
    names=[p for p in output.iterdir() if p.is_file()]
    receipts={p.name:{'bytes':p.stat().st_size,'sha256':sha256_file(p)} for p in names}
    value={'status':'complete','version':'cs781-final-public-refit-v1','split_sha256':split_hash,
           'labels':manifest['labels'],'parent_prepared_identity':manifest['identity'],'artifacts':receipts,
           'selection_policy':'Freeze settings/duration on original validation before refit; no refit validation score'}
    value['identity']=identity(value);atomic_json(output/'prepared_manifest.json',value)
    return output,value
''')


[Markdown Cell]

### state.py

Two recovery slots and best snapshot retain model, Adam, AMP scaler, scheduler, cursor and both ranks’ RNG. Corrupt files can fall back to another intact state. An all-invalid recovery tree stops rather than silently restarting.

[Markdown Cell]

Per-rank reproducibility and successful-update scheduling — source part 1

In [55]:
# [Code Cell]
append_module('state.py', r'''# CELL: Per-rank reproducibility and successful-update scheduling
import math
import os
from pathlib import Path
import random
import numpy as np
import torch
from .io_utils import atomic_json, ensure_space, publish_copy, read_json, sha256_file

def seed_everything(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False

def capture_rng(device: torch.device) -> dict:
    numpy_state = np.random.get_state()
    return {'python':random.getstate(), 'numpy':[numpy_state[0],numpy_state[1].tolist(),*numpy_state[2:]],
            'torch':torch.get_rng_state(), 'cuda':torch.cuda.get_rng_state(device) if device.type=='cuda' else None}

def restore_rng(state: dict, device: torch.device) -> None:
    random.setstate(state['python'])
    value=state['numpy']; np.random.set_state((value[0],np.asarray(value[1],dtype=np.uint32),*value[2:]))
    torch.set_rng_state(state['torch'])
    if device.type=='cuda':
        torch.cuda.set_rng_state(state['cuda'],device)

class WarmupCosine:
    """Schedule advances after successful optimizer updates, including on resume."""
    def __init__(self, optimizer, total_steps: int, warmup_fraction: float, minimum: float):
        self.optimizer, self.total_steps, self.minimum = optimizer,total_steps,minimum
        self.warmup_steps=max(1,int(total_steps*warmup_fraction))
        self.base_lrs=[group['lr'] for group in optimizer.param_groups]; self.steps=0
        self.apply()

    def apply(self) -> None:
        step=self.steps+1
        if step <= self.warmup_steps:
            scale=step/self.warmup_steps
        else:
            progress=min(1.0,(step-self.warmup_steps)/max(1,self.total_steps-self.warmup_steps))
            scale=self.minimum+(1-self.minimum)*0.5*(1+math.cos(math.pi*progress))
        for group,base in zip(self.optimizer.param_groups,self.base_lrs):
            group['lr']=base*scale

    def advance(self) -> None:
        self.steps+=1; self.apply()

''')


[Markdown Cell]

Per-rank reproducibility and successful-update scheduling — source part 2

In [56]:
# [Code Cell]
append_module('state.py', r'''    def state_dict(self) -> dict:
        return dict(steps=self.steps,total_steps=self.total_steps,warmup_steps=self.warmup_steps,
                    minimum=self.minimum,base_lrs=self.base_lrs)

    def load_state_dict(self, value: dict) -> None:
        for key in ('total_steps','warmup_steps','minimum','base_lrs'):
            if value[key] != getattr(self,key):
                raise ValueError(f'Scheduler plan changed: {key}')
        self.steps=value['steps']; self.apply()

''')


[Markdown Cell]

Two atomic full-state recovery slots plus a validation-best snapshot — source part 1

In [57]:
# [Code Cell]
append_module('state.py', r'''# CELL: Two atomic full-state recovery slots plus a validation-best snapshot
class Checkpoints:
    """Commit receipts last; select the latest intact state, preserving read-only inputs."""
    def __init__(self, run: Path, fingerprint: str, reserve: int, extra_roots: list[str] | None = None):
        self.directory=run/'checkpoints'; self.directory.mkdir(parents=True,exist_ok=True)
        self.fingerprint,self.reserve,self.generation=fingerprint,reserve,0
        self.roots=[self.directory,*[Path(root)/'checkpoints' for root in (extra_roots or [])]]
        self.warnings=[]

    def candidates(self, best_only: bool = False) -> list[tuple[Path,dict]]:
        result,seen=[],False
        for root in self.roots:
            for name in (('best.pt',) if best_only else ('slot_0.pt','slot_1.pt','best.pt')):
                path=root/name; sidecar=root/(name+'.json')
                if not path.exists() and not sidecar.exists():
                    continue
                seen=True
                try:
                    receipt=read_json(sidecar)
                    if receipt.get('fingerprint') != self.fingerprint or receipt.get('version') != 1:
                        raise ValueError('Experiment/source/Torch fingerprint mismatch')
                    if path.stat().st_size != receipt['bytes'] or sha256_file(path) != receipt['sha256']:
                        raise ValueError('Checkpoint size/SHA mismatch')
                    result.append((path,receipt)); self.generation=max(self.generation,receipt['generation'])
                except (ValueError,KeyError,OSError) as error:
                    self.warnings.append(f'{path}: {error}')
        if seen and not result:
            raise ValueError('All visible NER checkpoints failed verification. Refusing to silently restart. '+ '; '.join(self.warnings))
        return result

    def latest(self) -> tuple[Path,dict] | None:
        choices=self.candidates()
        if not choices:
            return None
        winner=max(choices,key=lambda pair:(pair[1]['step'],pair[1]['generation']))
        tied=[receipt['sha256'] for _,receipt in choices if (receipt['step'],receipt['generation'])==
              (winner[1]['step'],winner[1]['generation'])]
        if len(set(tied))>1:
            raise ValueError('Conflicting recovery states at the same step. Attach one exact saved version.')
        return winner

    def best(self) -> tuple[Path,dict] | None:
        values=self.candidates(best_only=True)
        return max(values,key=lambda pair:(pair[1]['validation_macro_f1'],pair[1]['step'])) if values else None

    def adopt(self, candidate: tuple[Path,dict], best: bool = False) -> tuple[Path,dict]:
        source,receipt=candidate
        destination=self.directory/('best.pt' if best else f"slot_{receipt['generation']%2}.pt")
''')


[Markdown Cell]

Two atomic full-state recovery slots plus a validation-best snapshot — source part 2

In [58]:
# [Code Cell]
append_module('state.py', r'''        if source.resolve()!=destination.resolve():
            ensure_space(self.directory,source.stat().st_size,self.reserve)
            publish_copy(source,destination)
            if sha256_file(destination)!=receipt['sha256']:
                raise ValueError('Restaged checkpoint checksum mismatch.')
            atomic_json(destination.with_name(destination.name+'.json'),receipt)
        return destination,receipt

    def load(self, candidate: tuple[Path,dict]) -> dict:
        path,receipt=candidate
        self.generation=max(self.generation,receipt['generation'])
        payload=torch.load(path,map_location='cpu',weights_only=True,mmap=True)
        required={'version','fingerprint','config','model','optimizer','scaler','scheduler','epoch','step','cursor','rng_by_rank'}
        if required-payload.keys() or payload['fingerprint']!=self.fingerprint:
            raise ValueError('Incomplete/incompatible NER checkpoint.')
        cursor=payload['cursor']
        if payload['epoch']!=receipt['epoch'] or payload['step']!=receipt['step'] or cursor['global_step']!=payload['step']:
            raise ValueError('Checkpoint cursor and receipt disagree.')
        if payload['scheduler']['steps']!=cursor['optimizer_steps'] or len(payload['rng_by_rank'])!=payload['config']['world_size']:
            raise ValueError('Missing/misaligned scheduler or per-rank RNG state.')
        return payload

    def commit(self, payload: dict, best: bool = False) -> tuple[Path,dict]:
        self.generation+=1
        path=self.directory/f'slot_{self.generation%2}.pt'
        temporary=path.with_name(path.name+'.partial')
        estimate=sum(tensor.numel()*tensor.element_size() for tensor in payload['model'].values())*3+(16<<20)
        ensure_space(self.directory,estimate,self.reserve)
        with temporary.open('wb') as stream:
            torch.save(payload,stream); stream.flush(); os.fsync(stream.fileno())
        os.replace(temporary,path)
        receipt={'version':1,'fingerprint':self.fingerprint,'sha256':sha256_file(path),'bytes':path.stat().st_size,
                 'epoch':payload['epoch'],'step':payload['step'],'generation':self.generation,
                 'validation_macro_f1':payload['cursor']['best_macro_f1']}
        atomic_json(path.with_name(path.name+'.json'),receipt)
        if best:
            publish_copy(path,self.directory/'best.pt')
            atomic_json(self.directory/'best.pt.json',receipt)
        return path,receipt
''')


[Markdown Cell]

### study.py

Finite candidate bundles test learned character context, boundaries and optimization; retain an actual stronger baseline.

[Markdown Cell]

Small, finite candidate set; method and original split stay fixed — source part 1

In [59]:
# [Code Cell]
append_module('study.py', r'''# CELL: Small, finite candidate set; method and original split stay fixed
"""Measure stronger learned context without a promised or fabricated F1 gain."""
import argparse
from dataclasses import asdict,replace
from pathlib import Path
import shutil
import time
from .config import NERConfig
from .artifacts import prepared_input
from .io_utils import atomic_json,identity,read_json,run_lock,sha256_file
from .workflow import (choose_task,TaskUnavailable,relocate_task,train_stage,
                       ensure_teacher,package_prediction,observations,execute)


def candidate_configs(base: NERConfig) -> list[tuple[str,NERConfig]]:
    """Two honest candidate bundles, not a claim of isolated factor effects."""
    first=replace(base,character_context=True,boundary_loss_weight=.2,
                  max_epochs=20,minimum_epochs=5,patience=4,stop_early=True,
                  additional_epochs=0,target_total_epochs=None,resume_run_dir=None)
    adapter=base.method in ('lora','lorma','adapters')
    second=replace(first,head_width=256,class_weight_max=10.,head_lr=5e-4,
                   backbone_lr=3e-4 if adapter else 3e-5,
                   adapter_rank=32 if base.method in ('lora','lorma') else base.adapter_rank,
                   adapter_alpha=64. if base.method in ('lora','lorma') else base.adapter_alpha,
                   bottleneck_width=128 if base.method=='adapters' else base.bottleneck_width,
                   unfreeze_top_layers=6 if base.method=='partial' else base.unfreeze_top_layers,
                   distillation_weight=.3 if base.method=='distillation' else base.distillation_weight,
                   head_dropout=.2 if base.family=='a' else .1)
    return [('character_context',first),('capacity_optimization',second)]


''')


[Markdown Cell]

Keep the best actually completed run, including an existing baseline — source part 1

In [60]:
# [Code Cell]
append_module('study.py', r'''# CELL: Keep the best actually completed run, including an existing baseline
def run_study(plan: dict,catalog_path: Path) -> dict:
    root=Path(plan['run_dir']);root.mkdir(parents=True,exist_ok=True)
    cfg=NERConfig(**plan['settings']);cfg.validate()
    prepared=prepared_input(replace(cfg,prepared_dir=plan.get('prepared_dir')))
    cfg=replace(cfg,prepared_dir=str(prepared))
    catalog=read_json(catalog_path)
    started=time.monotonic();results=[];paused=False
    try:baseline=choose_task(cfg,plan.get('ner_run_dir'),exclude_prefix=plan['experiment_id']+'_')
    except TaskUnavailable:baseline=None
    if baseline:
        workspace=root/'baseline_selection';workspace.mkdir(exist_ok=True)
        _,selected,outcome=relocate_task(baseline,prepared,workspace,catalog)
        reference=outcome['reference']
        if reference['parent_checkpoint_sha256']!=plan['expected_parent_checkpoint_sha256']:
            raise ValueError('Baseline uses a different own Phase 1 foundation')
        export=root/'baseline_export';export.mkdir(exist_ok=True)
        archive=package_prediction(outcome['native_test_predictions'],export,'submission_baseline_native.zip')
        results.append({'candidate':'existing_baseline','run_dir':str(baseline),
             'validation_macro_f1':reference['native_validation_macro_f1'],
             'zip_path':str(archive),'zip_sha256':sha256_file(archive)})
        cfg=replace(selected,run_dir=str(root),prepared_dir=str(prepared),
            input_root=cfg.input_root,parent_run_dir=cfg.parent_run_dir,
            tokenizer_path=cfg.tokenizer_path,metadata_path=cfg.metadata_path,
            teacher_run_dir=cfg.teacher_run_dir,max_run_seconds=cfg.max_run_seconds)
    cfg,teacher=ensure_teacher(cfg,plan,catalog_path,root)
    frozen={'settings':cfg.choices(),'budget_seconds':plan['budget_seconds'],
            'parent_sha256':plan['expected_parent_checkpoint_sha256'],
            'candidates':[(name,candidate.choices()) for name,candidate in candidate_configs(cfg)]}
    study_id=identity(frozen);old=root/'study_plan.json'
    if old.exists() and read_json(old)['identity']!=study_id:
        raise ValueError('Study configuration/baseline changed; reattach the same baseline or use a new study ID')
    atomic_json(old,{'identity':study_id,**frozen})
    if teacher and teacher['status']!='complete':
        report={'status':'paused_recoverable','teacher_bootstrap':teacher,'results':results}
        atomic_json(root/'study_report.json',report);observations(root);return report
    choices=candidate_configs(cfg)
    if not baseline:choices=[('fresh_control',replace(cfg,max_epochs=15)),*choices]
    for name,candidate in choices:
        remaining=int(plan['budget_seconds']-(time.monotonic()-started))
        if remaining<180:paused=True;break
        candidate=replace(candidate,experiment_id=plan['experiment_id']+'_'+name,
            run_dir=str(root/'trials'/name),max_run_seconds=min(candidate.max_run_seconds,remaining))
        value=train_stage(candidate,catalog_path,plan['expected_parent_checkpoint_sha256'])
        if value['status']!='complete':paused=True;break
        receipt=value['receipt']
        train_path=Path(candidate.run_dir)/'predictions/train_metrics.json'
        training=read_json(train_path)['strict_macro_f1'] if train_path.is_file() else None
''')


[Markdown Cell]

Keep the best actually completed run, including an existing baseline — source part 2

In [61]:
# [Code Cell]
append_module('study.py', r'''        results.append({'candidate':name,'run_dir':value['run_dir'],
            'validation_macro_f1':receipt['validation_macro_f1'],'train_macro_f1':training,
            'best_epoch':receipt['checkpoint_epoch'],'checkpoint_sha256':receipt['checkpoint_sha256'],
            'zip_path':receipt['zip_path'],'zip_sha256':receipt['zip_sha256']})
        atomic_json(root/'study_results.json',{'identity':study_id,'results':results})
    report={'status':'paused_recoverable' if paused else 'complete','results':results,
        'selection_metric':'Native strict Macro-F1 on original ten held-out documents',
        'candidate_effects_are_bundled':True,'private_test_labels_used':False,
        'scope':'New candidate improvements are unmeasured until actual training; no minimum F1 guarantee'}
    if results:
        winner=max(results,key=lambda row:(row['validation_macro_f1'],row['candidate']=='existing_baseline'))
        report['winner']=winner
        target=root/('submission_'+plan['experiment_id']+'_selected.zip')
        shutil.copyfile(winner['zip_path'],target)
        assert sha256_file(target)==winner['zip_sha256']
        report['submission_zip']=str(target)
        report['meets_validation_0_4']=winner['validation_macro_f1']>=.4
        remaining=int(plan['budget_seconds']-(time.monotonic()-started))
        if plan.get('refit_winner',False) and not paused and remaining>=180:
            final_plan={**plan,'mode':'refit70','run_dir':str(root/'winner_final'),
                        'experiment_id':plan['experiment_id']+'_winner',
                        'ner_run_dir':winner['run_dir'],'settings':{**asdict(cfg),'max_run_seconds':remaining}}
            report['refit']=execute(final_plan,catalog_path)
    atomic_json(root/'study_report.json',report);observations(root)
    print('Study winner:',report.get('winner'),flush=True)
    return report


def main():
    parser=argparse.ArgumentParser();parser.add_argument('--plan',required=True);parser.add_argument('--catalog',required=True)
    args=parser.parse_args();plan=read_json(args.plan)
    from .prepare import require_hardware
    require_hardware()
    with run_lock(Path(plan['run_dir'])):run_study(plan,Path(args.catalog).resolve())


if __name__=='__main__':main()
''')


[Markdown Cell]

### train.py

Two spawned T4 workers use NCCL DDP/FP16. Rank 0 reports epochs/steps/loss/LR. The loss is globally normalized over owned supervised characters; unsupervised dummy tail rows synchronize without dropping windows. Save every 500 updates and each epoch. Restore the original scheduler horizon and hold its final LR when extending.

[Markdown Cell]

Distributed runtime and task optimizer — source part 1

In [62]:
# [Code Cell]
append_module('train.py', r'''# CELL: Distributed runtime and task optimizer
from contextlib import nullcontext
from datetime import timedelta
import argparse
import json
import math
import os
from pathlib import Path
import signal
import socket
import time
import numpy as np
import torch
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from .config import NERConfig
from .data import WindowDataset, RankBatchSampler, collate_windows
from .io_utils import atomic_json, read_json, run_lock
from .metrics import filter_spans, metrics
from .model import build_model, forward_batch, weighted_loss, boundary_objective
from .predict import predict_partition, validate_predictions, official_validation
from .state import Checkpoints, WarmupCosine, capture_rng, restore_rng, seed_everything

def make_optimizer(model, cfg):
    groups=[{'params':[parameter for name,parameter in model.named_parameters() if name.startswith('backbone.') and parameter.requires_grad],
             'lr':cfg.backbone_lr},
            {'params':[parameter for name,parameter in model.named_parameters() if not name.startswith('backbone.') and parameter.requires_grad],
             'lr':cfg.head_lr}]
    return torch.optim.AdamW(groups,weight_decay=cfg.weight_decay,betas=(0.9,0.999),eps=1e-8,fused=True)

def initial_cursor() -> dict:
    return {'epoch':0,'next_batch':0,'global_step':0,'optimizer_steps':0,'history':[],
            'best_macro_f1':-1.0,'best_threshold':0.0,'bad_epochs':0,'complete':False,
            'epoch_loss_sum':0.0,'epoch_weight_sum':0.0,'epoch_characters':0,
            'epoch_overflows':0,'consecutive_overflows':0}

def stopped(device: torch.device, event, started: float, limit: int) -> bool:
    value=torch.tensor(int(event.is_set() or time.monotonic()-started>=limit),device=device)
    dist.all_reduce(value,op=dist.ReduceOp.MAX)
    return bool(value.item())

''')


[Markdown Cell]

Collective full-state checkpoint and consistent held-out selection — source part 1

In [63]:
# [Code Cell]
append_module('train.py', r'''# CELL: Collective full-state checkpoint and consistent held-out selection
def save_all(rank: int, manager, ddp, optimizer, scaler, scheduler, cursor, cfg, job, device, best=False, paused=False):
    rng=capture_rng(device); gathered=[None]*cfg.world_size if rank==0 else None
    dist.gather_object(rng,gathered,dst=0)
    if rank==0:
        payload={'version':1,'fingerprint':job['fingerprint'],'config':job['config'],
                 'model':ddp.module.state_dict(),'optimizer':optimizer.state_dict(),'scaler':scaler.state_dict(),
                 'scheduler':scheduler.state_dict(),'epoch':cursor['epoch'],'step':cursor['global_step'],
                 'cursor':cursor,'rng_by_rank':gathered}
        _,receipt=manager.commit(payload,best=best)
        atomic_json(Path(cfg.run_dir)/'training_summary.json',{
            'status':'complete' if cursor['complete'] else ('paused_at_saved_update' if paused else 'in_progress_checkpointed'),
            'epoch':cursor['epoch'],'next_batch':cursor['next_batch'],'step':cursor['global_step'],
            'maximum_epochs':cfg.max_epochs,'early_stopping':cursor['complete'] and cursor['epoch']<cfg.max_epochs,
            'optimizer_steps':cursor['optimizer_steps'],'best_validation_macro_f1':None if cfg.selection_mode=='public_refit' else cursor['best_macro_f1'],
            'history':cursor['history'],'checkpoint_sha256':receipt['sha256'],'fingerprint':job['fingerprint']})
    dist.barrier()

def validate_epoch(rank: int, model, job: dict, cfg, device) -> dict:
    if cfg.selection_mode=='public_refit':
        return {'threshold':cfg.fixed_confidence_threshold,'strict_macro_f1':None,
                'per_label':{},'examples':0,'scope':'No held-out metric after full public refit'}
    local=predict_partition(model,job,cfg,'validation',device,rank,cfg.world_size)
    collected=[None]*cfg.world_size if rank==0 else None
    dist.gather_object(local,collected,dst=0)
    result=None
    if rank==0:
        scored={key:value for part in collected for key,value in part.items()}
        gold=read_json(Path(job['prepared_dir'])/'validation_gold.json')
        trials=[]
        for threshold in cfg.confidence_thresholds:
            predictions=filter_spans(scored,threshold)
            validate_predictions(predictions,Path(job['prepared_dir']),'validation',job['labels'])
            value=metrics(gold,predictions,job['labels'])
            trials.append({'threshold':threshold,**value})
        result=max(trials,key=lambda row:(row['strict_macro_f1'],row['strict_macro_recall'],-row['threshold']))
        path=Path(cfg.run_dir)/'validation_prediction.json'
        atomic_json(path,filter_spans(scored,result['threshold']))
        official=official_validation(job,path,result)
        (Path(cfg.run_dir)/'official_validation_latest.txt').write_text(official,encoding='utf-8')
        atomic_json(Path(cfg.run_dir)/'validation_threshold_trials.json',{'trials':trials})
        atomic_json(Path(cfg.run_dir)/'validation_metrics.json',result)
    wrapper=[result]; dist.broadcast_object_list(wrapper,src=0)
    return wrapper[0]

''')


[Markdown Cell]

Restore the exact optimizer, schedule, RNG and next batch — source part 1

In [64]:
# [Code Cell]
append_module('train.py', r'''# CELL: Restore the exact optimizer, schedule, RNG and next batch
def worker(rank: int, job: dict, port: int, stop_event) -> None:
    cfg=NERConfig(**{**job['config'],'confidence_thresholds':tuple(job['config']['confidence_thresholds'])})
    torch.set_num_threads(2); torch.cuda.set_device(rank); device=torch.device('cuda',rank)
    dist.init_process_group(backend='nccl',init_method=f'tcp://127.0.0.1:{port}',
                            rank=rank,world_size=cfg.world_size,timeout=timedelta(minutes=30))
    try:
        seed_everything(cfg.seed)
        manager=Checkpoints(Path(cfg.run_dir),job['fingerprint'],cfg.min_free_bytes,job['resume_roots'])
        path=Path(job['resume_checkpoint']) if job.get('resume_checkpoint') else None
        receipt=read_json(path.with_name(path.name+'.json')) if path else None
        payload=manager.load((path,receipt)) if path else None
        model=build_model(job,cfg,device,load_parent=payload is None)
        if payload:
            model.load_state_dict(payload['model'],strict=True)
        ddp=DDP(model,device_ids=[rank],broadcast_buffers=False,find_unused_parameters=False)
        optimizer=make_optimizer(model,cfg)
        windows=WindowDataset(job['cache']['directory'],'train',1,job.get('teacher',{}).get('directory'))
        global_batch=cfg.world_size*cfg.micro_batch_size*cfg.accumulation_steps
        epoch_steps=math.ceil(len(windows)/global_batch)
        scheduler=WarmupCosine(optimizer,epoch_steps*(cfg.refit_schedule_epochs if cfg.selection_mode=='public_refit' else cfg.max_epochs),cfg.warmup_fraction,cfg.min_lr_ratio)
        if payload:
            saved_plan=payload['scheduler']
            scheduler.total_steps=saved_plan['total_steps']; scheduler.warmup_steps=saved_plan['warmup_steps']
            scheduler.minimum=saved_plan['minimum']; scheduler.base_lrs=saved_plan['base_lrs']
        scaler=torch.amp.GradScaler('cuda',init_scale=16384)
        cursor=initial_cursor()
        if payload:
            optimizer.load_state_dict(payload['optimizer']); scaler.load_state_dict(payload['scaler'])
            scheduler.load_state_dict(payload['scheduler']); cursor=payload['cursor']
            if not 0<=cursor['epoch']<=cfg.max_epochs or not 0<=cursor['next_batch']<=epoch_steps:
                raise ValueError('Saved NER epoch/batch cursor is invalid.')
            if cursor['global_step'] != cursor['epoch']*epoch_steps+cursor['next_batch']:
                raise ValueError('NER cursor does not match the full-corpus batch plan.')
            restore_rng(payload['rng_by_rank'][rank],device); del payload
        else:
            seed_everything(cfg.seed+rank)
            # Protect even sessions interrupted before the first 500-update save.
            save_all(rank,manager,ddp,optimizer,scaler,scheduler,cursor,cfg,job,device)
        if cursor['complete'] and cursor['epoch']<cfg.max_epochs and job['training_plan']['extension_requested']:
            cursor['complete']=False; cursor['bad_epochs']=0
        if cursor['complete']:
            save_all(rank,manager,ddp,optimizer,scaler,scheduler,cursor,cfg,job,device)
            if rank==0:
                print('Fixed final refit is complete; its validation-selected duration remains frozen.' if cfg.selection_mode=='public_refit' else 'This NER experiment is complete. Set additional_epochs in a fresh session or explicitly raise target_total_epochs to extend it.',flush=True)
            return
        weights=torch.tensor(job['cache']['class_weights'],device=device)
        boundary_weights=torch.tensor(job['cache']['boundary_class_weights'],device=device)
''')


[Markdown Cell]

Restore the exact optimizer, schedule, RNG and next batch — source part 2

In [65]:
# [Code Cell]
append_module('train.py', r'''        window_weights=np.load(Path(job['cache']['directory'])/'window_weights.npy',mmap_mode='r')
        window_boundary_weights=np.load(Path(job['cache']['directory'])/'window_boundary_weights.npy',mmap_mode='r')
        started=time.monotonic()
        while cursor['epoch']<cfg.max_epochs:
            epoch=cursor['epoch']; start_batch=cursor['next_batch']
            planner=RankBatchSampler(len(windows),cfg,rank,epoch,start_batch)
            loader_args=dict(batch_sampler=planner,collate_fn=collate_windows,pin_memory=True,
                             num_workers=cfg.num_workers,generator=torch.Generator().manual_seed(cfg.seed+epoch+rank))
            if cfg.num_workers:
                loader_args.update(multiprocessing_context='spawn',prefetch_factor=2,persistent_workers=False)
            loader=DataLoader(windows,**loader_args); iterator=iter(loader)
            ddp.train()
            progress=tqdm(total=epoch_steps,initial=start_batch,disable=rank!=0,
                          desc=f'{cfg.experiment_id} epoch {epoch+1}/{cfg.max_epochs}',mininterval=2)
            for batch_index in range(start_batch,epoch_steps):
                if stopped(device,stop_event,started,cfg.max_run_seconds):
                    save_all(rank,manager,ddp,optimizer,scaler,scheduler,cursor,cfg,job,device,paused=True)
                    progress.close(); return
                normalizer=float(window_weights[planner.global_indices(batch_index)].sum())
                assert normalizer>0
                boundary_norms=window_boundary_weights[planner.global_indices(batch_index)].sum(axis=0)
                normalizers=(normalizer,float(boundary_norms[0]),float(boundary_norms[1]))
                assert min(normalizers)>0
                optimizer.zero_grad(set_to_none=True)
                local_loss=torch.zeros((),device=device,dtype=torch.float64); local_characters=0
                for micro in range(cfg.accumulation_steps):
                    batch=next(iterator)
                    targets=batch['targets'].to(device,non_blocking=True)
                    local_characters+=int((batch['targets']!=-100).sum())
                    context=ddp.no_sync() if micro+1<cfg.accumulation_steps else nullcontext()
                    with context:
                        with torch.autocast('cuda',dtype=torch.float16):
                            outputs=forward_batch(ddp,batch,device,auxiliary=bool(cfg.boundary_loss_weight))
                            logits=outputs[0] if isinstance(outputs,tuple) else outputs
                            loss=boundary_objective(outputs,batch,weights,boundary_weights,normalizers,cfg,device)
                            if cfg.method=='distillation':
                                from .distill import distillation_loss
                                loss=(1-cfg.distillation_weight)*loss+cfg.distillation_weight*distillation_loss(
                                    logits,batch['teacher_logits'].to(device,non_blocking=True),targets,weights,cfg.distillation_temperature)
                        local_loss+=loss.detach().double()
                        scaler.scale(loss*(cfg.world_size/normalizer)).backward()
                stats=torch.stack((local_loss,torch.tensor(local_characters,device=device,dtype=torch.float64)))
                dist.all_reduce(stats)
                if not bool(torch.isfinite(stats).all()):
                    raise FloatingPointError('Nonfinite training loss. Restart from the last committed checkpoint; no partial gradients are saved.')
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(ddp.parameters(),cfg.max_grad_norm)
                old_scale=scaler.get_scale(); scaler.step(optimizer); scaler.update()
''')


[Markdown Cell]

Restore the exact optimizer, schedule, RNG and next batch — source part 3

In [66]:
# [Code Cell]
append_module('train.py', r'''                success=scaler.get_scale()>=old_scale
                if success:
                    scheduler.advance(); cursor['optimizer_steps']+=1; cursor['consecutive_overflows']=0
                else:
                    cursor['epoch_overflows']+=1; cursor['consecutive_overflows']+=1
                if cursor['consecutive_overflows']>cfg.max_consecutive_overflows:
                    raise FloatingPointError('Repeated AMP overflow. Resume from a committed checkpoint and inspect data/precision before changing settings.')
                cursor['global_step']+=1; cursor['next_batch']=batch_index+1
                cursor['epoch_loss_sum']+=float(stats[0]); cursor['epoch_weight_sum']+=normalizer
                cursor['epoch_characters']+=int(stats[1])
                optimizer.zero_grad(set_to_none=True)
                if rank==0:
                    progress.update(1); progress.set_postfix(loss=cursor['epoch_loss_sum']/cursor['epoch_weight_sum'],
                        lr=optimizer.param_groups[0]['lr'],step=cursor['global_step'],
                        val_f1='fixed_refit' if cfg.selection_mode=='public_refit' else max(0,cursor['best_macro_f1']),amp_skips=cursor['epoch_overflows'])
                if cursor['global_step']%cfg.checkpoint_every_steps==0:
                    save_all(rank,manager,ddp,optimizer,scaler,scheduler,cursor,cfg,job,device)
            progress.close(); del iterator,loader
            # An evaluator failure must not force replay of an already trained epoch.
            save_all(rank,manager,ddp,optimizer,scaler,scheduler,cursor,cfg,job,device)
            value=validate_epoch(rank,ddp.module,job,cfg,device)
            row={'epoch':epoch+1,'weighted_character_objective':cursor['epoch_loss_sum']/cursor['epoch_weight_sum'],
                 'loss_includes_distillation':cfg.method=='distillation',
                 'loss_includes_boundary_auxiliaries':bool(cfg.boundary_loss_weight),'head_context':cfg.head_context,
                 'training_characters':cursor['epoch_characters'],'overflows':cursor['epoch_overflows'],**value}
            expected_characters=sum(entry['characters'] for entry in job['cache']['documents']['train'])
            if cursor['epoch_characters']!=expected_characters:
                raise ValueError('Epoch did not cover every assigned training character exactly once.')
            cursor['history'].append(row)
            improved=cfg.selection_mode=='public_refit' or value['strict_macro_f1']>cursor['best_macro_f1']+1e-12
            if improved:
                cursor['best_macro_f1']=-1.0 if cfg.selection_mode=='public_refit' else value['strict_macro_f1']; cursor['best_threshold']=value['threshold']; cursor['bad_epochs']=0
            else:
                cursor['bad_epochs']+=1
            cursor['epoch']=epoch+1; cursor['next_batch']=0
            cursor['epoch_loss_sum']=0.0; cursor['epoch_weight_sum']=0.0; cursor['epoch_characters']=0; cursor['epoch_overflows']=0
            cursor['complete']=cursor['epoch']>=cfg.max_epochs or (job['training_plan']['early_stopping_active']
                                and cursor['epoch']>=cfg.minimum_epochs and cursor['bad_epochs']>=cfg.patience)
            save_all(rank,manager,ddp,optimizer,scaler,scheduler,cursor,cfg,job,device,best=improved)
            if rank==0:
                print(json.dumps({'epoch':cursor['epoch'],'validation_macro_f1':value['strict_macro_f1'],
                                  'threshold':value['threshold'],'best':None if cfg.selection_mode=='public_refit' else cursor['best_macro_f1'],'complete':cursor['complete']}),flush=True)
            if cursor['complete']:
                return
    finally:
        dist.destroy_process_group()

''')


[Markdown Cell]

Importable two-T4 launcher and graceful interruption — source part 1

In [67]:
# [Code Cell]
append_module('train.py', r'''# CELL: Importable two-T4 launcher and graceful interruption
def main() -> None:
    parser=argparse.ArgumentParser(); parser.add_argument('--job',required=True)
    args=parser.parse_args(); job=read_json(args.job)
    cfg=NERConfig(**{**job['config'],'confidence_thresholds':tuple(job['config']['confidence_thresholds'])})
    cfg.validate()
    from .prepare import require_hardware
    require_hardware()
    os.environ.setdefault('TOKENIZERS_PARALLELISM','false')
    os.environ.setdefault('TORCH_NCCL_ASYNC_ERROR_HANDLING','1')
    with run_lock(Path(cfg.run_dir)):
        manager=Checkpoints(Path(cfg.run_dir),job['fingerprint'],cfg.min_free_bytes,job['resume_roots'])
        latest=manager.latest(); best=manager.best()
        if best:
            manager.adopt(best,best=True)
        if latest:
            latest=manager.adopt(latest)
            job['resume_checkpoint']=str(latest[0])
            print(f'Resume saved step {latest[1]["step"]}, completed epochs {latest[1]["epoch"]}',flush=True)
        else:
            job['resume_checkpoint']=None
            print('Start NER from verified own Phase 1 weights; new head/Adam/scaler/scheduler.',flush=True)
        atomic_json(Path(cfg.run_dir)/'resume_report.json',{'checkpoint':job['resume_checkpoint'],'warnings':manager.warnings})
        context=mp.get_context('spawn'); event=context.Event()
        signal.signal(signal.SIGINT,lambda *_:event.set()); signal.signal(signal.SIGTERM,lambda *_:event.set())
        with socket.socket() as server:
            server.bind(('127.0.0.1',0)); port=server.getsockname()[1]
        processes=mp.spawn(worker,args=(job,port,event),nprocs=2,join=False)
        while not processes.join(timeout=1):
            pass

if __name__=='__main__':
    main()
''')


[Markdown Cell]

### workflow.py

Auto starts a missing supervised task from verified own Phase 1. Explicit recovery paths and corruption checks remain strict.

[Markdown Cell]

Verified existing-task selection and explicit finalization modes — source part 1

In [68]:
# [Code Cell]
append_module('workflow.py', r'''# CELL: Verified existing-task selection and explicit finalization modes
"""Calibrate existing own models, or cold-refit their frozen recipes on public data."""
import argparse
from dataclasses import asdict,replace
import json
import os
from pathlib import Path
import shutil
import signal
import subprocess
import sys
import torch
import zipfile
from .config import NERConfig
from .io_utils import atomic_json,identity,read_json,records,sha256_file,run_lock
from .parents import source_hashes,parent_candidates
from .artifacts import prepared_input
from .data import build_cache
from .model import build_model
from .predict import predict_partition,validate_predictions
from .metrics import metrics,filter_spans
from .state import Checkpoints
from .calibration import calibrate,apply_policy
from .public_data import public_refit_bundle
from .distill import LEGACY_SOURCE_SHA256,EXPERIMENT_SOURCE_SHA256,FINAL_SOURCE_SHA256


class TaskUnavailable(FileNotFoundError):
    """No completed task was attached; auto mode can train one from own Phase 1."""


def choose_task(cfg: NERConfig,explicit: str | None=None,exclude_prefix: str | None=None) -> Path:
    """Select only completed held-out tasks of the declared family AND method."""
    paths=[Path(explicit)/'job.json'] if explicit else sorted(Path(cfg.input_root).rglob('job.json'))
    candidates=[]
    if explicit and not (Path(explicit)/'job.json').is_file():
        raise FileNotFoundError('ner_run_dir must be the task/trial folder containing job.json. Clear it to None for Phase-1-only auto training.')
    for path in paths:
        job=read_json(path);saved=job.get('config',{})
        if exclude_prefix and saved.get('experiment_id','').startswith(exclude_prefix):continue
        if explicit and (saved.get('family')!=cfg.family or saved.get('method')!=cfg.method):
            raise ValueError('Explicit ner_run_dir belongs to another family/method')
        if saved.get('family')!=cfg.family or saved.get('method')!=cfg.method:continue
        if saved.get('selection_mode','heldout')!='heldout':continue
        receipt_path=path.parent/'submission_receipt.json';summary_path=path.parent/'training_summary.json'
        if not receipt_path.exists() or not summary_path.exists():continue
        receipt=read_json(receipt_path);summary=read_json(summary_path)
        if summary.get('status')!='complete' or receipt.get('status')!='submission_verified':continue
''')


[Markdown Cell]

Verified existing-task selection and explicit finalization modes — source part 2

In [69]:
# [Code Cell]
append_module('workflow.py', r'''        if not isinstance(receipt.get('validation_macro_f1'),(int,float)):continue
        candidates.append((receipt['validation_macro_f1'],receipt['checkpoint_epoch'],path.parent))
    if not candidates:
        raise TaskUnavailable(f'No completed held-out {cfg.family.upper()} / {cfg.method} task attached. Use auto/develop mode with complete own Phase 1 output, or attach the trained task with job.json and checkpoint/receipt/source/tokenizer files.')
    return max(candidates,key=lambda row:(row[0],row[1],str(row[2])))[2]


def relocate_task(root: Path,original: Path,workspace: Path,catalog: dict) -> tuple[dict,NERConfig,dict]:
    """Verify actual task source/tokenizer/state independently of former paths."""
    job=read_json(root/'job.json');version=job['version']
    packages={'cs781-flexible-ner-v1':'ner_flexible','cs781-ner-experiments-v2':'ner_experiments','cs781-ner-final-v1':'ner_final','cs781-ner-repair-v2':'ner_repair'}
    if version not in packages:raise ValueError('Unsupported saved-task runtime')
    expected={'cs781-flexible-ner-v1':LEGACY_SOURCE_SHA256,'cs781-ner-experiments-v2':EXPERIMENT_SOURCE_SHA256,'cs781-ner-final-v1':FINAL_SOURCE_SHA256}.get(version,source_hashes(Path(__file__).parent))
    if source_hashes(root/'code'/packages[version])!=expected or job['runtime_source_sha256']!=expected:
        raise ValueError('Incomplete/changed saved-task source')
    if str(torch.__version__)!=job['environment']['torch']:raise ValueError('Use the recorded Kaggle PyTorch build '+job['environment']['torch'])
    cfg=NERConfig(**{**job['config'],'confidence_thresholds':tuple(job['config']['confidence_thresholds'])})
    if cfg.selection_mode!='heldout':raise ValueError('Full-public refit cannot supply a held-out selection score')
    manifest=read_json(original/'prepared_manifest.json')
    if cfg.expected_split_sha256!=manifest['split_sha256'] or job['labels']!=manifest['labels']:
        raise ValueError('Saved task uses a different selection split or label order')
    cfg=replace(cfg,run_dir=str(workspace),prepared_dir=str(original))
    source=root/'parent_source'/f'model_{cfg.family}';token=root/'tokenizer/tokenizer.json';meta=root/'tokenizer/metadata.json'
    if source_hashes(source)!=catalog[cfg.family]['source_sha256']:raise ValueError('Parent architecture source changed')
    if sha256_file(token)!=job['parent']['tokenizer_sha256'] or sha256_file(meta)!=job['parent']['metadata_sha256']:
        raise ValueError('Saved tokenizer or metadata changed')
    value={**job,'parent_source_dir':str(source),'tokenizer_path':str(token),'prepared_dir':str(original),
           'parent_contract':catalog[cfg.family]}
    value['cache']=build_cache(original,token,cfg,job['parent'])
    manager=Checkpoints(workspace,job['fingerprint'],cfg.min_free_bytes,[str(root)])
    candidate=manager.best();receipt=read_json(root/'submission_receipt.json')
    if candidate is None or candidate[1]['sha256']!=receipt['checkpoint_sha256']:raise ValueError('Best state and submission receipt disagree')
    payload=manager.load(candidate)
    device=torch.device('cuda',0);torch.set_num_threads(2)
    model=build_model(value,cfg,device,load_parent=False);model.load_state_dict(payload['model'],strict=True)
    threshold=float(payload['cursor']['best_threshold']);del payload
    scored=predict_partition(model,value,cfg,'validation',device)
    gold=read_json(original/'validation_gold.json')
    native=metrics(gold,filter_spans(scored,threshold),value['labels'])
    if abs(native['strict_macro_f1']-receipt['validation_macro_f1'])>1e-6:
        raise ValueError('Loaded model does not reproduce the saved validation score')
    calibration=calibrate(scored,gold,value['labels'],threshold)
    atomic_json(workspace/'calibration_report.json',calibration)
    test_scored=predict_partition(model,value,cfg,'test',device)
    test=apply_policy(test_scored,calibration['policy']);validate_predictions(test,original,'test',value['labels'])
    del model;torch.cuda.empty_cache()
    reference={'root':str(root),'checkpoint_sha256':receipt['checkpoint_sha256'],'native_validation_macro_f1':native['strict_macro_f1'],
        'config':asdict(cfg),'parent_checkpoint_sha256':job['parent']['checkpoint_sha256'],
''')


[Markdown Cell]

Verified existing-task selection and explicit finalization modes — source part 3

In [70]:
# [Code Cell]
append_module('workflow.py', r'''        'best_epoch':receipt['checkpoint_epoch'],'calibration':calibration,
        'teacher_sha256':job.get('teacher',{}).get('teacher_sha256'),
        'selection_uses_private_scores':False,'test_gold_used':False}
    return value,cfg,{'reference':reference,'test_predictions':test,
                      'native_test_predictions':filter_spans(test_scored,threshold)}


''')


[Markdown Cell]

One-root-file submissions and small reproducible reports — source part 1

In [71]:
# [Code Cell]
append_module('workflow.py', r'''# CELL: One-root-file submissions and small reproducible reports
def package_prediction(prediction: dict,root: Path,name: str) -> Path:
    path=root/'prediction.json';atomic_json(path,prediction)
    archive=root/name;temporary=archive.with_name(archive.name+'.partial')
    with zipfile.ZipFile(temporary,'w',zipfile.ZIP_DEFLATED) as z:z.write(path,'prediction.json')
    with zipfile.ZipFile(temporary) as z:
        if z.namelist()!=['prediction.json'] or z.testzip() is not None:raise ValueError('Invalid final submission ZIP')
    os.replace(temporary,archive);return archive


def stage_runtime(stage: Path) -> Path:
    package=stage/'code/ner_repair';package.mkdir(parents=True,exist_ok=True)
    expected=source_hashes(Path(__file__).parent)
    if (stage/'job.json').exists() and source_hashes(package)!=expected:raise ValueError('Exact stage resume requires its original source')
    for path in Path(__file__).parent.glob('*.py'):shutil.copyfile(path,package/path.name)
    return package.parent


def run_module(code: Path,module: str,*args) -> None:
    env=os.environ.copy();env['PYTHONPATH']=str(code)+os.pathsep+env.get('PYTHONPATH','')
    process=subprocess.Popen([sys.executable,'-u','-m','ner_repair.'+module,*map(str,args)],cwd=code,env=env)
    previous={s:signal.getsignal(s) for s in (signal.SIGINT,signal.SIGTERM)}
    def stop(signum,_):
        if process.poll() is None:process.send_signal(signum)
    try:
        for signum in previous:signal.signal(signum,stop)
        result=process.wait()
        if result:raise subprocess.CalledProcessError(result,process.args)
    finally:
        for signum,handler in previous.items():signal.signal(signum,handler)


def train_stage(cfg: NERConfig,catalog_path: Path,expected_parent: str | None,policy: dict | None=None,expected_teacher: str | None=None) -> dict:
    root=Path(cfg.run_dir);code=stage_runtime(root)
    launch=root/'launch_config.json';atomic_json(launch,asdict(cfg))
    run_module(code,'prepare','--config',launch,'--catalog',catalog_path)
    job=read_json(root/'job.json')
    if expected_parent and job['parent']['checkpoint_sha256']!=expected_parent:raise ValueError('Wrong Phase 1 foundation for frozen recipe')
    if expected_teacher and job.get('teacher',{}).get('teacher_sha256')!=expected_teacher:
        raise ValueError('Attach the same teacher used by the selected KD development task')
    if policy:atomic_json(root/'frozen_calibration.json',policy)
    run_module(code,'train','--job',root/'job.json')
    summary=read_json(root/'training_summary.json')
    if summary['status']!='complete':return {'status':'paused_recoverable','run_dir':str(root)}
    run_module(code,'predict','--job',root/'job.json')
    return {'status':'complete','run_dir':str(root),'receipt':read_json(root/'submission_receipt.json')}


''')


[Markdown Cell]

One-root-file submissions and small reproducible reports — source part 2

In [72]:
# [Code Cell]
append_module('workflow.py', r'''def ensure_teacher(cfg: NERConfig,plan: dict,catalog_path: Path,root: Path) -> tuple[NERConfig,dict | None]:
    """Reuse a held-out teacher, or explicitly bootstrap own A when available."""
    if cfg.method!='distillation':return cfg,None
    if cfg.teacher_run_dir:
        if not (Path(cfg.teacher_run_dir)/'job.json').is_file():
            raise FileNotFoundError('teacher_run_dir must contain the trained teacher job.json and weights')
        return cfg,None
    paths=list(Path(cfg.input_root).rglob('job.json'))+list(root.rglob('job.json'))
    for path in paths:
        value=read_json(path);settings=value.get('config',{})
        summary=path.parent/'training_summary.json'
        if (settings.get('family') in ('a','c','d') and settings.get('selection_mode','heldout')=='heldout'
            and summary.is_file() and read_json(summary).get('status')=='complete'):
            # prepare_teacher_cache independently verifies source, size, tensors,
            # selection split and teacher > student size before using a candidate.
            if path.parent.is_relative_to(root):return replace(cfg,teacher_run_dir=str(path.parent)),None
            return cfg,None
    if not plan.get('allow_teacher_bootstrap',True):
        raise FileNotFoundError('KD needs a completed own A/C/D NER teacher. Attach one or enable own-A bootstrap with its full Phase 1 output.')
    teacher=replace(cfg,family='a',method='full',experiment_id=plan['experiment_id']+'_teacher_development',
        run_dir=str(root/'teacher_development'),prepared_dir=cfg.prepared_dir,
        parent_run_dir=plan.get('teacher_parent_run_dir'),tokenizer_path=None,metadata_path=None,
        resume_run_dir=None,teacher_run_dir=None,head_context='local',character_context=False,
        head_width=128,boundary_loss_weight=.2,backbone_lr=2e-5,head_lr=3e-4,
        additional_epochs=0,target_total_epochs=None,max_epochs=15,seed=783)
    catalog=read_json(catalog_path)
    has_saved=(Path(teacher.run_dir)/'job.json').is_file() or any(
        read_json(p).get('config',{}).get('experiment_id')==teacher.experiment_id for p in paths)
    if not has_saved and len([r for r in parent_candidates(teacher,catalog['a']) if r['accepted']])!=1:
        raise FileNotFoundError('To bootstrap KD attach ONE complete own Phase 1 A output plus A/B matching tokenizer/metadata, or attach a trained A/C/D Phase 2 teacher. B Phase 1 weights alone cannot supply a teacher.')
    print('No task-trained teacher attached: train/restore own A on the original 60/10 split first.',flush=True)
    result=train_stage(teacher,catalog_path,plan.get('teacher_parent_checkpoint_sha256'))
    return replace(cfg,teacher_run_dir=result['run_dir']),result


def observations(root: Path) -> Path:
    archive=root/'final_observations.zip'
    allowed={'final_report.json','calibration_report.json','final_reference.json','development_reference.json',
             'study_report.json','study_results.json','study_plan.json','pipeline_plan.json','submission_receipt.json',
             'training_summary.json','task_config.json','job.json','frozen_calibration.json','data_audit.json',
             'train_metrics.json','validation_metrics.json','validation_unfiltered_metrics.json'}
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
        for path in root.rglob('*.json'):
            if path.name in allowed:z.write(path,path.relative_to(root))
    with zipfile.ZipFile(archive) as z:
        if z.testzip() is not None:raise ValueError('Observation ZIP corrupted')
    return archive

''')


[Markdown Cell]

One-root-file submissions and small reproducible reports — source part 3

In [73]:
# [Code Cell]
append_module('workflow.py', r'''
''')


[Markdown Cell]

Freeze on development, then refit without validation feedback — source part 1

In [74]:
# [Code Cell]
append_module('workflow.py', r'''# CELL: Freeze on development, then refit without validation feedback
def execute(plan: dict,catalog_path: Path) -> dict:
    root=Path(plan['run_dir']);root.mkdir(parents=True,exist_ok=True)
    mode=plan['mode']
    if mode not in ('auto','develop','inference','refit70','develop_then_refit'):raise ValueError('Unknown mode')
    cfg=NERConfig(**plan['settings']);cfg.validate()
    original=prepared_input(replace(cfg,run_dir=str(root),prepared_dir=plan.get('prepared_dir')))
    catalog=read_json(catalog_path);report={'mode':mode,'family':cfg.family,'method':cfg.method,'test_gold_used':False}
    wants_development=mode in ('develop','develop_then_refit')
    saved_id=plan['experiment_id']+'_development'
    saved_local=(root/'development/job.json').is_file()
    saved_input=any(read_json(p).get('config',{}).get('experiment_id')==saved_id
                    for p in Path(cfg.input_root).rglob('job.json'))
    task=None
    if mode=='auto':
        if cfg.additional_epochs or cfg.target_total_epochs is not None or saved_local or saved_input:
            wants_development=True
        else:
            try:task=choose_task(cfg,plan.get('ner_run_dir'))
            except TaskUnavailable as error:
                report['auto_start_reason']=str(error);wants_development=True
                print('No Phase 2 task attached. Auto mode starts supervised development from verified own Phase 1 weights.',flush=True)
    if wants_development:
        cfg,teacher_result=ensure_teacher(replace(cfg,prepared_dir=str(original)),plan,catalog_path,root)
        if teacher_result:
            report['teacher_bootstrap']=teacher_result
            if teacher_result['status']!='complete':
                report['status']='paused_recoverable';atomic_json(root/'final_report.json',report)
                observations(root);return report
        development=replace(cfg,run_dir=str(root/'development'),experiment_id=plan['experiment_id']+'_development',
                            prepared_dir=str(original),selection_mode='heldout',fixed_final_epochs=0)
        result=train_stage(development,catalog_path,plan.get('expected_parent_checkpoint_sha256'))
        report['development']=result
        if result['status']!='complete':
            report['status']='paused_recoverable';atomic_json(root/'final_report.json',report)
            observations(root);return report
        task=Path(result['run_dir'])
    elif task is None:task=choose_task(cfg,plan.get('ner_run_dir'))
    workspace=root/'selection';workspace.mkdir(exist_ok=True)
    job,selected,outcome=relocate_task(task,original,workspace,catalog)
    reference=outcome['reference']
    if plan.get('expected_parent_checkpoint_sha256') and reference['parent_checkpoint_sha256']!=plan['expected_parent_checkpoint_sha256']:
        raise ValueError('Selected NER task used a different Phase 1 parent')
    cold=replace(selected,experiment_id=plan['experiment_id']+'_refit70',run_dir=str(root/'refit70'),
        prepared_dir=None,parent_run_dir=cfg.parent_run_dir,tokenizer_path=cfg.tokenizer_path,metadata_path=cfg.metadata_path,
        parent_checkpoint=cfg.parent_checkpoint,resume_run_dir=None,teacher_run_dir=cfg.teacher_run_dir,
        max_run_seconds=cfg.max_run_seconds)
    frozen={'checkpoint_sha256':reference['checkpoint_sha256'],'parent_sha256':reference['parent_checkpoint_sha256'],
''')


[Markdown Cell]

Freeze on development, then refit without validation feedback — source part 2

In [75]:
# [Code Cell]
append_module('workflow.py', r'''            'teacher_sha256':reference['teacher_sha256'],
            'epoch_count':reference['best_epoch'],'learning_choices':selected.choices(),
            'calibration':reference['calibration']['policy'],'selection_split_sha256':selected.expected_split_sha256}
    frozen_hash=identity(frozen);reference['frozen_recipe_sha256']=frozen_hash
    old=root/('development_reference.json' if mode=='develop' else 'final_reference.json')
    if mode!='develop' and old.exists() and read_json(old)['frozen_recipe_sha256']!=frozen_hash:
        raise ValueError('Frozen recipe changed; select a new final experiment ID')
    atomic_json(old,reference)
    calibrated=package_prediction(outcome['test_predictions'],root,'submission_'+plan['experiment_id']+'_calibrated.zip')
    report.update(source=reference,calibrated_development_zip=str(calibrated))
    if mode in ('inference','develop'):
        report.update(status='complete',submission_zip=str(calibrated),validation_score_scope='Development calibration fit, not a new independently held-out score')
    else:
        public,manifest=public_refit_bundle(original,root/'all_public_prepared')
        final=replace(cold,prepared_dir=str(public),expected_prepared_identity=manifest['identity'],expected_split_sha256=manifest['split_sha256'],
            selection_mode='public_refit',selection_split_sha256=selected.expected_split_sha256,selection_reference_sha256=frozen_hash,
            fixed_final_epochs=reference['best_epoch'],max_epochs=reference['best_epoch'],stop_early=False,
            additional_epochs=0,target_total_epochs=None,refit_schedule_epochs=selected.max_epochs,
            fixed_confidence_threshold=reference['calibration']['policy']['global_threshold'])
        final.validate()
        result=train_stage(final,catalog_path,reference['parent_checkpoint_sha256'],reference['calibration'],reference['teacher_sha256'])
        report['refit']=result;report['status']=result['status']
        if result['status']=='complete':
            report['submission_zip']=result['receipt']['zip_path'];report['refit_validation_macro_f1']=None
            report['refit_score_scope']='All 70 released training documents; no held-out validation remains. Private score unmeasured.'
    atomic_json(root/'final_report.json',report);report['observations_zip']=str(observations(root))
    return report


def main():
    parser=argparse.ArgumentParser();parser.add_argument('--plan',required=True);parser.add_argument('--catalog',required=True)
    args=parser.parse_args();plan=read_json(args.plan)
    from .prepare import require_hardware
    require_hardware()
    with run_lock(Path(plan['run_dir'])):
        print(json.dumps(execute(plan,Path(args.catalog).resolve()),indent=2),flush=True)


if __name__=='__main__':main()
''')


In [76]:
# [Code Cell]
EXPECTED_MODULE_SHA256={'__init__.py': 'a6ee8e23898c6dd802a7c3285bd4c9cef008152a8dbac95b7ddbd2219b42876f', 'adaptation.py': 'b5f452257274cf43e41eb0c489807cf8e643977419584b301245d469d4a0e2ed', 'artifacts.py': 'ad515bd2ee897afb9c8a98c90b14b422fcb751802837e50929b8f667aaf24b2c', 'calibration.py': 'fe111bb74274908034d3cb9d161cd35934f36c83fe80745dc42bd2a55fa5112d', 'config.py': '3cd77a380b2b21c5befde7ef0e8f5c93b7d5c2a6f3866bc835304cf421e95992', 'data.py': 'd7b6fabbf6f24395f35b5f7a7e2b790401b9d34a33a6a0e0fe8fa0976e6e6332', 'distill.py': '454ca6f6909f3c3ec491f62f358571ef1fa4a6d556e0901b63fed7d5fcff28d4', 'io_utils.py': '3f9d828342c819690d390f0280c01f9eda160aab59c9cf424d6b3c2a31813e73', 'metrics.py': 'e5e611b0c176543531ddbd97b9f3b6f0bd4d34b3b27d430a71befb2fdabd1dbb', 'model.py': 'f599e13cd3de0ca11c0b5944531a2d178d4d28cc10101ab784494b4e5e25e9de', 'parents.py': '834710b94c3f82eb18b48abf752727e30d245ebbe58f55408263b54fdddcd660', 'predict.py': 'd519ca116eb43342102ed7982dd08f8f707667ac7d4f80ebbf0a26d9c82fddc7', 'prepare.py': '193edcd90569f255582392c6e68102a021363d8b6c7cd7f4b717996834503987', 'public_data.py': 'a6eec9fb24e66d8f5280e4618e0ce6214e525cbf48751d9b30c2b9a4bd56f9be', 'state.py': 'f05cc134dcbff7b59050460dccc6d61cbd5b75ad6920175526b459ef3a1eeebf', 'study.py': '3bf2f176ca2d3f8d298c9e2ce70362b5e5faeba56de7f5074a2659d8e2637c11', 'train.py': 'c0cbb46c8866bb13a175a6b4e5b10aee0a8f45a163eea86f100ed7195bcd0885', 'workflow.py': '90a46250cb5e8a305b4cbd86a04430711adc28b124aab6c829344b1e2fb43dc0'}


[Markdown Cell]

## 5. Materialize importable workers and create the plan

In [77]:
# [Code Cell]
PARENT_CATALOG={'a': {'family': 'a', 'runtime_version': 'cs781-model-a-v1', 'source_sha256': {'config.py': 'c0d1e9e7eb49a11e12ff0d7014cd07c6b90149460da5db3b06e9af71fbb3f390', 'model.py': '05118a8c9eec3e58adb272554e7356e81fb28c95cbe7905f39ebd5ee2d670319', 'data.py': '39b20ec14101e66b6c1f81464738161a71755f53f30fec7edb93deb7f4ffe9a8', 'state.py': '723193f75a39d3b8f16af1e6f1bfbd507b1775a0cd2430583d211e54e69d63c2', 'train.py': '3ea14c27742ab82e686083a669823678529d0f8beb46b2e6b2113f0aef452743', 'infer.py': '1cb8e96f2d49aa1c0b25c8e0ef6420c6d63f17ec8b72813d69b646748fb46013', 'sanity.py': '7716a61318767f9001ac08cdcafbe228f53c3c0d6f8fb9cc3806353ec2d6577f', '__init__.py': '59a82a09276925aa3342bc0991d79d474388cc6119bae880ced9d2403eb20a45'}, 'fingerprint_exclusions': ['data_dir', 'run_dir', 'resume_dir', 'initial_checkpoint', 'max_run_seconds', 'min_free_bytes', 'max_checkpoint_bytes', 'generation_examples_per_source', 'generation_token_limit'], 'data_fingerprint': '3571562850214ac1b423548cb0020fdcf1a3ffd8956e0368c456ec1b100292f4', 'audited_original_epochs': 2, 'audited_original_step': 29942, 'audited_original_fingerprint': '24f90bc6e148ffb53dcf6132138b69a95ced2e6a5ca34690601d29def059cddd', 'recorded_torch_candidates': ['2.11.0+cu128', '2.10.0+cu128'], 'class_name': 'LegalCausalDecoder'}, 'b': {'family': 'b', 'runtime_version': 'cs781-model-b-v1', 'source_sha256': {'config.py': '71407468d5c4b105b5db2c533fb736eb847b4b67cdf9d4f2ab014436adef1e9b', 'model.py': '09baae8e7ebe0508fb149a64c187975b7a86ae62376b5bc2ec4cef2fb2471a99', 'data.py': 'd01b01d43f5c4c24bb0429f7bbb8bbbd4c52d52c8787bd5ef3f40bdac28df919', 'state.py': '026036ddbdf43e949755da263eb97378855a28281129c6a63f3b3ea69abbe088', 'train.py': 'f928bb52eaafd17dc061d3003326fd13cf2f79b5ccbc226f6af0177429aefab0', 'infer.py': '80ee8fff95675dccb59cf19da091355020495c0f41ea16fdb17d8f24357a59f4', 'sanity.py': '6b1b96bc16a4d7f0945e139fd60f35813b0bf9d2b075c4fd893a87d432fc9212', '__init__.py': '3dfc8da18d4c5d73d0ff1b52dc572672f8095b605dcc86836832d77607749087'}, 'fingerprint_exclusions': ['data_dir', 'run_dir', 'resume_dir', 'max_run_seconds', 'min_free_bytes', 'max_checkpoint_bytes', 'generation_examples_per_source', 'generation_token_limit'], 'data_fingerprint': '3571562850214ac1b423548cb0020fdcf1a3ffd8956e0368c456ec1b100292f4', 'audited_original_epochs': 3, 'audited_original_step': 24267, 'audited_original_fingerprint': 'e686c0f98fb5fe8fc03553d0151961dd6af0dd52ac041753de0502521a922bb1', 'recorded_torch_candidates': ['2.11.0+cu128', '2.10.0+cu128'], 'class_name': 'LegalEncoderDecoder'}, 'c': {'family': 'c', 'runtime_version': 'cs781-model-c-v1', 'source_sha256': {'config.py': 'f9fea0bc87f74ccac761f7d4dc574c4d1315a459523dba21849279c17bc24ace', 'model.py': '6f91835c789d12aadb5355bc06050b8b6c1fc231cd16126fc39f31769706aad4', 'data.py': '39b20ec14101e66b6c1f81464738161a71755f53f30fec7edb93deb7f4ffe9a8', 'state.py': 'b3a3ed7109fdfe43ebc8e75db89f2916d33e95ebe901e99d3e4aa9b6e8422ee4', 'train.py': '9bf6cb023ad1c8fdc4fbb652b4b3b6d3551ae777df139184051f8d23f4e5c0c9', 'infer.py': '062010c87b66c7badcef8f7a62ef3a89fdc7f96149d317214b13a7e679f40b6a', 'sanity.py': 'c56b6ad692b90cb5afb23a03a75a26dde9d9b2c9ef3e14ca2ed37cb8b0f18a68', '__init__.py': '59a82a09276925aa3342bc0991d79d474388cc6119bae880ced9d2403eb20a45'}, 'fingerprint_exclusions': ['data_dir', 'run_dir', 'resume_dir', 'initial_checkpoint', 'max_run_seconds', 'min_free_bytes', 'max_checkpoint_bytes', 'generation_examples_per_source', 'generation_token_limit'], 'data_fingerprint': '092deede43ffd17c00e5d11e1f4aa3a443b7939af4504c488426b70c53d1ed54', 'audited_original_epochs': 1, 'audited_original_step': 7383, 'audited_original_fingerprint': '8febd61da7f1d32f78eee1cd4a865d72e5a5b3a0068739210db323859c6d7caa', 'recorded_torch_candidates': ['2.11.0+cu128', '2.10.0+cu128'], 'class_name': 'LegalTransformerGRU'}, 'd': {'family': 'd', 'runtime_version': 'cs781-model-d-v1', 'source_sha256': {'config.py': '42bd096bdf40822a7cb1d84e7f437e3e64d2ef20b99ee81c884f60639378e63a', 'model.py': 'a2300d34524df50cb43a3ff700e72b3a8764f7566d66ba5065c048c5f115eae9', 'data.py': '39b20ec14101e66b6c1f81464738161a71755f53f30fec7edb93deb7f4ffe9a8', 'state.py': 'dee5a29320756f4b84ec349b24091180be65a8f4c86e7f16eb74343737f4e2db', 'train.py': 'efb53d8eb54a08ef1a4abc49d5437eb472ebf5047bf790cf2f041c17fd6cef37', 'infer.py': 'aa3b849b0095bbdb5d81d3bfafe6a8134d4481856ee7cb9db5fba2bd84b80b67', 'sanity.py': '8c02e1c1a1e521a5d1e83fd96b3cfba3fa9f7c76c5824f62d45297a4829ca298', '__init__.py': 'f49e11a2d96bae160f02704fd5818fe613e77534e06a0f16150dbc4bc2d4f5e6'}, 'fingerprint_exclusions': ['data_dir', 'run_dir', 'resume_dir', 'initial_checkpoint', 'max_run_seconds', 'min_free_bytes', 'max_checkpoint_bytes', 'generation_examples_per_source', 'generation_token_limit'], 'data_fingerprint': '3a4dc2be3ff16945069b15d7a2e351012d4a0c9319b95b5e4534cc6596e8693f', 'audited_original_epochs': 1, 'audited_original_step': 1828, 'audited_original_fingerprint': 'b9659899c46424ec56ad0607f32bd6e7097752a8066dd30a38764469e05c1352', 'recorded_torch_candidates': ['2.11.0+cu128', '2.10.0+cu128'], 'class_name': 'LegalSparseDecoder'}}


In [78]:
# [Code Cell]
import importlib
import hashlib
code_directory=Path(OUTPUT_ROOT)/'code'
package_directory=code_directory/'ner_repair'
package_directory.mkdir(parents=True,exist_ok=True)
for name,parts in module_parts.items():
    source=''.join(parts)
    assert hashlib.sha256(source.encode()).hexdigest()==EXPECTED_MODULE_SHA256[name],name
    compile(source,name,'exec')
    destination=package_directory/name
    protected=any((Path(OUTPUT_ROOT)/p).exists() for p in ('final_reference.json','development/job.json','study_plan.json'))
    if protected and destination.exists() and destination.read_text(encoding='utf-8')!=source:
        raise ValueError('Existing study requires its exact notebook runtime; use a new study ID')
    destination.write_bytes(source.encode('utf-8'))
for name in list(sys.modules):
    if name=='ner_repair' or name.startswith('ner_repair.'):del sys.modules[name]
sys.path.insert(0,str(code_directory));importlib.invalidate_caches()
catalog_path=code_directory/'parent_catalog.json'
catalog_path.write_text(json.dumps(PARENT_CATALOG,indent=2),encoding='utf-8')


5810

In [79]:
# [Code Cell]
from ner_repair.io_utils import atomic_json,read_json
SETTINGS=dict(family=FAMILY,method=METHOD,experiment_id=EXPERIMENT_ID,
    run_dir=OUTPUT_ROOT,prepared_dir=INPUTS['prepared_dir'],
    parent_run_dir=INPUTS['parent_run_dir'],tokenizer_path=INPUTS['tokenizer_path'],
    metadata_path=INPUTS['metadata_path'],teacher_run_dir=INPUTS['teacher_run_dir'],
    resume_run_dir=INPUTS['resume_development_dir'],
    max_epochs=DEVELOPMENT_EPOCHS,additional_epochs=ADDITIONAL_EPOCHS,
    target_total_epochs=TARGET_TOTAL_EPOCHS,
    seed=(783 if FAMILY=='a' and METHOD=='full' else 781),
    backbone_lr=(1e-4 if METHOD in ('lora','lorma','adapters') else 2e-5),head_lr=3e-4,
    head_width=128,head_context=('token_bigru' if FAMILY=='b' and METHOD=='lora' else 'local'),
    character_context=CHARACTER_CONTEXT,boundary_loss_weight=(.2 if FAMILY=='a' else 0.),
    micro_batch_size=2,accumulation_steps=4,
    max_run_seconds=90*60)
PLAN=dict(mode=RUN_MODE,run_dir=OUTPUT_ROOT,experiment_id=EXPERIMENT_ID,
    prepared_dir=INPUTS['prepared_dir'],ner_run_dir=INPUTS['ner_run_dir'],
    teacher_parent_run_dir=INPUTS['teacher_parent_run_dir'],
    allow_teacher_bootstrap=ALLOW_TEACHER_BOOTSTRAP,
    teacher_parent_checkpoint_sha256=KNOWN_PARENT_SHA256['a'],
    expected_parent_checkpoint_sha256=EXPECTED_PARENT_SHA256,settings=SETTINGS,
    budget_seconds=int(STUDY_BUDGET_HOURS*3600),refit_winner=REFIT_STUDY_WINNER)
plan_path=Path(OUTPUT_ROOT)/'pipeline_plan.json';atomic_json(plan_path,PLAN)
print('Mode:',RUN_MODE,'family/method:',FAMILY,METHOD)


Mode: develop family/method: b lora


[Markdown Cell]

## 6. Execute and recover

Two spawned NCCL workers train with FP16 and rank-0 progress. Full recovery states save initially, every 500 update boundaries and at epoch boundaries. A paused run resumes from its last committed update with the same source/settings/inputs. Development continuation uses its original saved schedule, holding the floor after its horizon.

In [80]:
# [Code Cell]
import codecs
import select
import signal

def run_streamed(command: list[str]) -> None:
    environment=os.environ.copy()
    environment['PYTHONPATH']=str(code_directory)+os.pathsep+environment.get('PYTHONPATH','')
    process=subprocess.Popen(command,cwd=code_directory,env=environment,stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT,bufsize=0)
    decoder=codecs.getincrementaldecoder('utf-8')('replace')
    assert process.stdout is not None
    try:
        while True:
            try:
                ready,_,_=select.select([process.stdout],[],[],1.0)
                if ready:
                    chunk=os.read(process.stdout.fileno(),8192)
                    if chunk:
                        sys.stdout.write(decoder.decode(chunk)); sys.stdout.flush()
                    elif process.poll() is not None:
                        break
                elif process.poll() is not None:
                    break
            except KeyboardInterrupt:
                print('Requesting a safe stop at the next update boundary; keep this cell running.')
                process.send_signal(signal.SIGINT)
        return_code=process.wait()
        sys.stdout.write(decoder.decode(b'',final=True)); sys.stdout.flush()
        if return_code:
            raise subprocess.CalledProcessError(return_code,command)
    finally:
        process.stdout.close()


In [81]:
# [Code Cell]
run_streamed([sys.executable,'-u','-m','ner_repair.workflow','--plan',str(plan_path),'--catalog',str(catalog_path)])


GPU 0: Tesla T4
GPU 1: Tesla T4
Verify prepared artifacts: 100%|██████████| 12/12 [00:00<00:00, 34.91it/s]
GPU 0: Tesla T4
GPU 1: Tesla T4
Verify prepared artifacts: 100%|██████████| 12/12 [00:00<00:00, 209.10it/s]
{
  "parent_candidates": [
    {
      "run_dir": "/kaggle/input/notebooks/shubhamjoshi1806/modelb-decoder/model_b_extended_run",
      "reasons": [],
      "accepted": true,
      "planned_epochs": 6,
      "completed_epochs": 6,
      "step": 48534,
      "status": "complete",
      "checkpoint_pairs": [
        "slot_0.pt",
        "slot_1.pt",
        "best.pt"
      ]
    }
  ]
}
Verified Model B: 6 completed epochs; selected best.pt at epoch 6
Cache all train documents: 60it [00:02, 23.26it/s]
Cache all validation documents: 10it [00:00, 18.56it/s]
Cache all test documents: 35it [00:01, 24.87it/s]
{
  "parent_completed_epochs": 6,
  "NER_plan": {
    "target_epoch": 15,
    "start_completed_epochs": 0,
    "extension_requested": false,
    "schedule_policy": "restore t

[Markdown Cell]

## 7. Read real results and download the submission

In [82]:
# [Code Cell]
report=read_json(Path(OUTPUT_ROOT)/'final_report.json')
print('Status:',report['status'])
print('Native development F1:',report.get('source',{}).get('native_validation_macro_f1'))
print('Study winner:',report.get('winner'))
print('Public-refit validation F1:',report.get('refit_validation_macro_f1'))
from IPython.display import display,FileLink
for key in ('submission_zip','calibrated_development_zip'):
    if report.get(key):display(FileLink(report[key]))
observations=Path(OUTPUT_ROOT)/'final_observations.zip'
if observations.exists():display(FileLink(str(observations)))
print('Keep the entire output for checkpoint recovery and teacher reuse.')


Status: complete
Native development F1: 0.15724594057518818
Study winner: None
Public-refit validation F1: None


/kaggle/working/cs781_phase2_rebuild/Rebuild_B_lora/submission_Rebuild_B_lora_calibrated.zip

/kaggle/working/cs781_phase2_rebuild/Rebuild_B_lora/submission_Rebuild_B_lora_calibrated.zip

/kaggle/working/cs781_phase2_rebuild/Rebuild_B_lora/final_observations.zip

Keep the entire output for checkpoint recovery and teacher reuse.


[Markdown Cell]

## Keep and reuse the complete result

Your learned task is OUTPUT_ROOT/development/. It contains job.json, task_config.json, training_summary.json, submission_receipt.json, checkpoint/receipt pairs, code/ner_repair/, parent_source/ and tokenizer/. The top-level output contains final_report.json, the displayed calibrated submission ZIP and final_observations.zip. Keep the **entire output**, including checkpoint files; the small observation ZIP alone cannot restore weights.

Use the displayed submission ZIP with one root prediction.json for Codabench. For later repaired final notebooks, set ner_run_dir to this exact development/ task. For notebook 06, set teacher_run_dir to notebook 01's development/ task. The old output names boundaries_seed783/context_seed781 are not required by these rebuilds.

To continue this development job, attach its whole saved output, keep this notebook/settings/EXPERIMENT_ID unchanged, and set resume_development_dir to its development/ folder. ADDITIONAL_EPOCHS=5 requests five more epochs beyond the saved plan once per working output; best validation weights are retained. The higher TARGET_TOTAL_EPOCHS alternative is also available. Both options cannot be set together. A change to LR/head/method is a new experiment with a new ID.

To train the frozen final recipe on all 70 public documents, use the corresponding repaired final notebook in refit70 mode, with a new final ID and ner_run_dir pointing to this completed held-out task. That final refit has no held-out validation F1. Do not choose settings using private test answers. New private scores are unmeasured until server evaluation.
